<a href="https://colab.research.google.com/github/mpdepauladasilva/Detection-Chagas-Disease/blob/dev/Estudo_Dirigido_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1-Realizar uma revisão bibliográfica

Sobre o estado da arte na detecção da Doença de Chagas via ECG
e as aplicações das arquiteturas de deep learning propostas (XLSTM, LSTM, GRU, Transformers, CNN,TCN).

In [ ]:
# Substitua 'caminho/para/seu/arquivo.zip' pelo caminho completo do arquivo zip no seu Google Drive.
# Substitua 'caminho/para/pasta/destino' pelo caminho completo da pasta onde você quer extrair os arquivos.
# Certifique-se de que o Google Drive esteja montado.

!unzip -n '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output.zip' -d '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/'

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19200_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19200_hr.dat  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19202_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19203_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19204_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19206_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19207_hr.hea  
  inflating: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/19000/19210_hr.dat  
  inflating: /content/drive/MyDrive/Doutorado

In [ ]:
!ls /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part0

A saída de streaming foi truncada nas últimas 5000 linhas.
1084292.dat  1416102.dat  2522382.dat  3082283.dat  4282.dat	 744671.dat
1084292.hea  1416102.hea  2522382.hea  3082283.hea  4282.hea	 744671.hea
1084381.dat  1416279.dat  2522894.dat  308228.dat   428375.dat	 744752.dat
1084381.hea  1416279.hea  2522894.hea  308228.hea   428375.hea	 744752.hea
108466.dat   141640.dat   2522899.dat  3082426.dat  428430.dat	 744784.dat
108466.hea   141640.hea   2522899.hea  3082426.hea  428430.hea	 744784.hea
1084672.dat  1416464.dat  2522915.dat  3082473.dat  428601.dat	 74479.dat
1084672.hea  1416464.hea  2522915.hea  3082473.hea  428601.hea	 74479.hea
1084691.dat  141652.dat   252367.dat   30824.dat    428774.dat	 744864.dat
1084691.hea  141652.hea   252367.hea   30824.hea    428774.hea	 744864.hea
1084801.dat  1416579.dat  252385.dat   3082787.dat  428824.dat	 744958.dat
1084801.hea  1416579.hea  252385.hea   3082787.hea  428824.hea	 744958.hea
1084896.dat  1416716.dat  2523903.dat  3082792.

# 2 -Adquirir e preparar os conjuntos de dados públicos



implementando uma rotina de pre-processamento que inclua:

• A normalização dos dados para o intervalo [0, 1];

In [ ]:
!pip install wfdb

In [ ]:
!ls -1 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output/ | wc -l

1630


In [ ]:
!ls -1 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/ | wc -l

18


In [ ]:
!ls -1 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output/ | wc -l

22


In [ ]:
# Crie a pasta de destino no armazenamento local temporário
!mkdir -p /content/chagas_datasets/

# Copie os dados do Drive para o armazenamento local da VM (muito mais rápido)
# Substitua os caminhos e comandos conforme necessário para cada dataset
!cp -r '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output' /content/chagas_datasets/
!cp -r '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output' /content/chagas_datasets/
!cp -r '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output' /content/chagas_datasets/

# Altere seus base_paths para usar o caminho local
base_paths = {
    'CODE-15%': '/content/chagas_datasets/code15_output',
    'PTB-XL': '/content/chagas_datasets/ptbxl_output',
    'SaMi-Trop': '/content/chagas_datasets/samitrop_output'
}

^C


# 3 -Organização do Dataset

In [ ]:
"""
Script SIMPLIFICADO para Extração de Informações de Headers .hea
Foco nas colunas: ['record_id', 'dataset', 'age', 'sex', 'chagas_label', 'file_path']
"""

import os
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def extract_simple_header_info(header_path, dataset_name):
    """Extrai apenas as informações essenciais do header"""
    try:
        # Leitura do arquivo .hea
        with open(header_path, 'r') as f:
            header_lines = f.readlines()

        if not header_lines:
            return None

        # Processa primeira linha para record_id
        first_line = header_lines[0].strip().split()
        record_id = first_line[0]

        # Extrai informações das comments
        age = None
        sex = None
        chagas_label = None

        for line in header_lines:
            if line.startswith('#'):
                comment = line[1:].strip()

                if 'Age:' in comment:
                    age_str = comment.split('Age:')[-1].strip().split()[0]
                    try:
                        age = int(age_str)
                    except:
                        age = None

                elif 'Sex:' in comment:
                    sex_str = comment.split('Sex:')[-1].strip().split()[0].lower()
                    if sex_str in ['female', 'f']:
                        sex = 'F'
                    elif sex_str in ['male', 'm']:
                        sex = 'M'

                elif 'Chagas label:' in comment:
                    chagas_str = comment.split('Chagas label:')[-1].strip().split()[0]
                    chagas_label = 1 if chagas_str.lower() == 'true' else 0

        # Define label padrão baseado no dataset se não encontrado
        if chagas_label is None:
            if dataset_name == 'SaMi-Trop':
                chagas_label = 1
            elif dataset_name == 'PTB-XL':
                chagas_label = 0

        return {
            'record_id': record_id,
            'dataset': dataset_name,
            'age': age,
            'sex': sex,
            'chagas_label': chagas_label,
            'file_path': str(header_path)
        }

    except Exception as e:
        return None

def process_dataset_simple(dataset_name, base_path):
    """Processa um dataset com barra de progresso"""
    if not os.path.exists(base_path):
        print(f"⚠️  Pasta não encontrada: {base_path}")
        return []

    print(f"🔍 Processando {dataset_name}...")

    # Encontra todos os arquivos .hea
    header_files = list(Path(base_path).rglob("*.hea"))
    print(f"   📁 Encontrados {len(header_files)} arquivos .hea")

    if not header_files:
        return []

    records = []

    # Processamento com barra de progresso
    with ThreadPoolExecutor() as executor:
        # Prepara as tarefas
        futures = {
            executor.submit(extract_simple_header_info, header, dataset_name): header
            for header in header_files
        }

        # Processa com barra de progresso
        for future in tqdm(as_completed(futures),
                          total=len(header_files),
                          desc=f"{dataset_name}"):
            result = future.result()
            if result is not None:
                records.append(result)

    print(f"✅ {dataset_name}: {len(records)} registros processados")
    return records

def main():
    """Função principal simplificada"""

    # Configurações dos datasets
    base_paths = {
        'CODE-15%': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output',
        'PTB-XL': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output',
        'SaMi-Trop': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output'
    }

    all_records = []
    start_time = time.time()

    # Processa cada dataset sequencialmente com barras de progresso individuais
    for dataset_name, base_path in base_paths.items():
        dataset_records = process_dataset_simple(dataset_name, base_path)
        all_records.extend(dataset_records)

    # Cria DataFrame apenas com as colunas desejadas
    df = pd.DataFrame(all_records)

    # Garante a ordem das colunas
    columns_order = ['record_id', 'dataset', 'age', 'sex', 'chagas_label', 'file_path']
    df = df[columns_order]

    if not df.empty:
        # Salva o resultado
        output_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/chagas_simple_dataset.csv'
        df.to_csv(output_path, index=False)

        elapsed_time = time.time() - start_time

        # Relatório final
        print(f"\n🎉 PROCESSAMENTO CONCLUÍDO!")
        print(f"⏱️  Tempo total: {elapsed_time:.2f} segundos")
        print(f"📊 Total de registros: {len(df):,}")
        print(f"📁 Arquivo salvo: {output_path}")

        # Estatísticas básicas
        print(f"\n📊 ESTATÍSTICAS:")
        print(f"Por dataset:")
        for dataset in df['dataset'].unique():
            count = len(df[df['dataset'] == dataset])
            print(f"   {dataset}: {count:,} registros")

        print(f"\nDistribuição de Chagas:")
        chagas_counts = df['chagas_label'].value_counts().sort_index()
        for label, count in chagas_counts.items():
            desc = "Positivo" if label == 1 else "Negativo"
            print(f"   {desc}: {count:,} registros")

        print(f"\nDistribuição por sexo:")
        sex_counts = df['sex'].value_counts()
        for sex, count in sex_counts.items():
            print(f"   {sex}: {count:,} registros")

        print(f"\nIdade:")
        print(f"   Mínima: {df['age'].min()} anos")
        print(f"   Máxima: {df['age'].max()} anos")
        print(f"   Média: {df['age'].mean():.1f} anos")

        print(f"\n📋 AMOSTRA DOS DADOS:")
        print(df.head().to_string(index=False))

    else:
        print("❌ Nenhum registro processado!")

    return df

# Executa o script
if __name__ == "__main__":
    print("🚀 Iniciando extração simplificada...")
    df = main()

🚀 Iniciando extração simplificada...
🔍 Processando CODE-15%...
   📁 Encontrados 326434 arquivos .hea


CODE-15%:  99%|█████████▉| 323350/326434 [9:34:11<06:08,  8.37it/s]

In [ ]:
import os
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import csv # Usado para salvar em modo append

# --- CONFIGURAÇÃO ---
OUTPUT_CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/chagas_ptbxl_samitrop_dataset.csv'
# Salva os novos dados a cada 1000 registros processados.
SAVE_INTERVAL = 1000
# --- CONFIGURAÇÃO ---

def extract_simple_header_info(header_path, dataset_name):
    """Extrai apenas as informações essenciais do header"""
    try:
        # Leitura do arquivo .hea
        with open(header_path, 'r') as f:
            header_lines = f.readlines()

        if not header_lines:
            return None

        # Processa primeira linha para record_id
        first_line = header_lines[0].strip().split()
        record_id = first_line[0]

        # Extrai informações das comments
        age = None
        sex = None
        chagas_label = None

        for line in header_lines:
            if line.startswith('#'):
                comment = line[1:].strip()

                if 'Age:' in comment:
                    age_str = comment.split('Age:')[-1].strip().split()[0]
                    try:
                        age = int(age_str)
                    except:
                        age = None

                elif 'Sex:' in comment:
                    sex_str = comment.split('Sex:')[-1].strip().split()[0].lower()
                    if sex_str in ['female', 'f']:
                        sex = 'F'
                    elif sex_str in ['male', 'm']:
                        sex = 'M'

                elif 'Chagas label:' in comment:
                    chagas_str = comment.split('Chagas label:')[-1].strip().split()[0]
                    chagas_label = 1 if chagas_str.lower() == 'true' else 0

        # Define label padrão baseado no dataset se não encontrado
        if chagas_label is None:
            if dataset_name == 'SaMi-Trop':
                chagas_label = 1
            elif dataset_name == 'PTB-XL':
                chagas_label = 0

        return {
            'record_id': record_id,
            'dataset': dataset_name,
            'age': age,
            'sex': sex,
            'chagas_label': chagas_label,
            'file_path': str(header_path)
        }

    except Exception as e:
        # Em caso de erro, retorna None
        return None

def save_checkpoint(records_to_save, output_path, header_exists):
    """Salva os novos registros no CSV de saída em modo append."""

    # Define as colunas desejadas para o CSV
    columns_order = ['record_id', 'dataset', 'age', 'sex', 'chagas_label', 'file_path']

    # Cria um DataFrame temporário
    df_temp = pd.DataFrame(records_to_save, columns=columns_order)

    # Escreve no arquivo CSV. Se o cabeçalho não existir, escreve o cabeçalho.
    df_temp.to_csv(output_path,
                   mode='a', # Modo append
                   header=not header_exists, # Escreve o cabeçalho se for a primeira vez
                   index=False,
                   quoting=csv.QUOTE_MINIMAL)

def process_dataset_simple(dataset_name, base_path, processed_ids, output_path, header_exists):
    """Processa um dataset com checkpointing."""
    if not os.path.exists(base_path):
        print(f"⚠️  Pasta não encontrada: {base_path}")
        return 0, header_exists

    print(f"🔍 Processando {dataset_name}...")

    # Encontra todos os arquivos .hea
    header_files = list(Path(base_path).rglob("*.hea"))

    # Filtra os arquivos que JÁ foram processados
    files_to_process = [
        f for f in header_files
        if f.stem not in processed_ids # f.stem é o nome do arquivo sem extensão, que é o record_id
    ]

    print(f"   📁 Total encontrados: {len(header_files)}. Ignorando {len(header_files) - len(files_to_process)} já processados.")

    if not files_to_process:
        return 0, header_exists

    newly_processed_count = 0
    records_buffer = []

    # Processamento com barra de progresso
    with ThreadPoolExecutor() as executor:
        # Prepara as tarefas apenas para os arquivos não processados
        futures = {
            executor.submit(extract_simple_header_info, header, dataset_name): header
            for header in files_to_process
        }

        # Processa com barra de progresso
        for future in tqdm(as_completed(futures),
                          total=len(files_to_process),
                          desc=f"{dataset_name}"):

            result = future.result()
            if result is not None:
                records_buffer.append(result)
                newly_processed_count += 1

                # Salva o checkpoint se o buffer atingir o intervalo de salvamento
                if len(records_buffer) >= SAVE_INTERVAL:
                    save_checkpoint(records_buffer, output_path, header_exists)
                    header_exists = True # O cabeçalho agora existe no arquivo
                    records_buffer = [] # Limpa o buffer

    # Salva o restante dos registros no buffer após o loop
    if records_buffer:
        save_checkpoint(records_buffer, output_path, header_exists)
        header_exists = True

    print(f"✅ {dataset_name}: {newly_processed_count} novos registros processados")
    return newly_processed_count, header_exists

def main_checkpoint():
    """Função principal com lógica de checkpointing."""

    # Configurações dos datasets
    base_paths = {
        'PTB-XL': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/ptbxl_output',
        'SaMi-Trop': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output'
    }

    start_time = time.time()
    total_processed = 0

    # 1. Carregar Checkpoint
    processed_ids = set()
    header_exists = os.path.exists(OUTPUT_CSV_PATH)

    if header_exists:
        try:
            # Carrega apenas a coluna 'record_id' para otimizar a memória
            existing_df = pd.read_csv(OUTPUT_CSV_PATH, usecols=['record_id'])
            processed_ids = set(existing_df['record_id'].astype(str).tolist())
            print(f"💾 Checkpoint encontrado: {len(processed_ids)} registros já processados.")

        except pd.errors.EmptyDataError:
            print("💾 Checkpoint encontrado, mas arquivo está vazio.")
            header_exists = False
        except Exception as e:
            print(f"❌ Erro ao ler checkpoint: {e}. Reiniciando processamento.")
            header_exists = False

    # 2. Processar cada dataset
    for dataset_name, base_path in base_paths.items():
        count, header_exists = process_dataset_simple(
            dataset_name,
            base_path,
            processed_ids,
            OUTPUT_CSV_PATH,
            header_exists
        )
        total_processed += count

    elapsed_time = time.time() - start_time

    # 3. Relatório final
    final_size = len(processed_ids) + total_processed

    print(f"\n🎉 PROCESSAMENTO CONCLUÍDO!")
    print(f"⏱️  Tempo total: {elapsed_time:.2f} segundos")
    print(f"🔄 Novos registros processados nesta execução: {total_processed:,}")
    print(f"📊 Total de registros no arquivo final: {final_size:,}")
    print(f"📁 Arquivo salvo: {OUTPUT_CSV_PATH}")

    # Retorna o DataFrame completo (opcional, só para análise final)
    if final_size > 0:
        return pd.read_csv(OUTPUT_CSV_PATH)
    return pd.DataFrame()

# Executa o script
if __name__ == "__main__":
    print("🚀 Iniciando extração com Checkpointing...")
    df = main_checkpoint()

    if not df.empty:
        # Estatísticas (Opcional: Re-executar o bloco de estatísticas aqui se desejar)
        print(f"\n📋 AMOSTRA DOS DADOS FINAIS:")
        print(df.tail().to_string(index=False))

🚀 Iniciando extração com Checkpointing...
🔍 Processando PTB-XL...
   📁 Total encontrados: 21799. Ignorando 0 já processados.


PTB-XL: 100%|██████████| 21799/21799 [16:27<00:00, 22.07it/s]


✅ PTB-XL: 21799 novos registros processados
🔍 Processando SaMi-Trop...
   📁 Total encontrados: 815. Ignorando 0 já processados.


SaMi-Trop: 100%|██████████| 815/815 [01:07<00:00, 12.16it/s] 


✅ SaMi-Trop: 815 novos registros processados

🎉 PROCESSAMENTO CONCLUÍDO!
⏱️  Tempo total: 1058.74 segundos
🔄 Novos registros processados nesta execução: 22,614
📊 Total de registros no arquivo final: 22,614
📁 Arquivo salvo: /content/drive/MyDrive/Doutorado-Material/Dataset/chagas_ptbxl_samitrop_dataset.csv

📋 AMOSTRA DOS DADOS FINAIS:
record_id   dataset  age sex  chagas_label                                                                                   file_path
    22083 SaMi-Trop   37   M             1  /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output/22083.hea
   386108 SaMi-Trop   69   M             1 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output/386108.hea
    65606 SaMi-Trop   44   F             1  /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output/65606.hea
   184107 SaMi-Trop   66   F             1 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/samitrop_output/

In [ ]:
import os
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import csv # Usado para salvar em modo append

# --- CONFIGURAÇÃO ---
OUTPUT_CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/code15_chagas_dataset.csv'
# Salva os novos dados a cada 1000 registros processados.
SAVE_INTERVAL = 1000
# --- CONFIGURAÇÃO ---

def extract_simple_header_info(header_path, dataset_name):
    """Extrai apenas as informações essenciais do header"""
    try:
        # Leitura do arquivo .hea
        with open(header_path, 'r') as f:
            header_lines = f.readlines()

        if not header_lines:
            return None

        # Processa primeira linha para record_id
        first_line = header_lines[0].strip().split()
        record_id = first_line[0]

        # Extrai informações das comments
        age = None
        sex = None
        chagas_label = None

        for line in header_lines:
            if line.startswith('#'):
                comment = line[1:].strip()

                if 'Age:' in comment:
                    age_str = comment.split('Age:')[-1].strip().split()[0]
                    try:
                        age = int(age_str)
                    except:
                        age = None

                elif 'Sex:' in comment:
                    sex_str = comment.split('Sex:')[-1].strip().split()[0].lower()
                    if sex_str in ['female', 'f']:
                        sex = 'F'
                    elif sex_str in ['male', 'm']:
                        sex = 'M'

                elif 'Chagas label:' in comment:
                    chagas_str = comment.split('Chagas label:')[-1].strip().split()[0]
                    chagas_label = 1 if chagas_str.lower() == 'true' else 0

        # Define label padrão baseado no dataset se não encontrado
        if chagas_label is None:
            if dataset_name == 'SaMi-Trop':
                chagas_label = 1
            elif dataset_name == 'PTB-XL':
                chagas_label = 0

        return {
            'record_id': record_id,
            'dataset': dataset_name,
            'age': age,
            'sex': sex,
            'chagas_label': chagas_label,
            'file_path': str(header_path)
        }

    except Exception as e:
        # Em caso de erro, retorna None
        return None

def save_checkpoint(records_to_save, output_path, header_exists):
    """Salva os novos registros no CSV de saída em modo append."""

    # Define as colunas desejadas para o CSV
    columns_order = ['record_id', 'dataset', 'age', 'sex', 'chagas_label', 'file_path']

    # Cria um DataFrame temporário
    df_temp = pd.DataFrame(records_to_save, columns=columns_order)

    # Escreve no arquivo CSV. Se o cabeçalho não existir, escreve o cabeçalho.
    df_temp.to_csv(output_path,
                   mode='a', # Modo append
                   header=not header_exists, # Escreve o cabeçalho se for a primeira vez
                   index=False,
                   quoting=csv.QUOTE_MINIMAL)

def process_dataset_simple(dataset_name, base_path, processed_ids, output_path, header_exists):
    """Processa um dataset com checkpointing."""
    if not os.path.exists(base_path):
        print(f"⚠️  Pasta não encontrada: {base_path}")
        return 0, header_exists

    print(f"🔍 Processando {dataset_name}...")

    # Encontra todos os arquivos .hea
    header_files = list(Path(base_path).rglob("*.hea"))

    # Filtra os arquivos que JÁ foram processados
    files_to_process = [
        f for f in header_files
        if f.stem not in processed_ids # f.stem é o nome do arquivo sem extensão, que é o record_id
    ]

    print(f"   📁 Total encontrados: {len(header_files)}. Ignorando {len(header_files) - len(files_to_process)} já processados.")

    if not files_to_process:
        return 0, header_exists

    newly_processed_count = 0
    records_buffer = []

    # Processamento com barra de progresso
    with ThreadPoolExecutor() as executor:
        # Prepara as tarefas apenas para os arquivos não processados
        futures = {
            executor.submit(extract_simple_header_info, header, dataset_name): header
            for header in files_to_process
        }

        # Processa com barra de progresso
        for future in tqdm(as_completed(futures),
                          total=len(files_to_process),
                          desc=f"{dataset_name}"):

            result = future.result()
            if result is not None:
                records_buffer.append(result)
                newly_processed_count += 1

                # Salva o checkpoint se o buffer atingir o intervalo de salvamento
                if len(records_buffer) >= SAVE_INTERVAL:
                    save_checkpoint(records_buffer, output_path, header_exists)
                    header_exists = True # O cabeçalho agora existe no arquivo
                    records_buffer = [] # Limpa o buffer

    # Salva o restante dos registros no buffer após o loop
    if records_buffer:
        save_checkpoint(records_buffer, output_path, header_exists)
        header_exists = True

    print(f"✅ {dataset_name}: {newly_processed_count} novos registros processados")
    return newly_processed_count, header_exists

def main_checkpoint():
    """Função principal com lógica de checkpointing."""

    # Configurações dos datasets
    base_paths = {
        'CODE-15%': '/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output'
    }

    start_time = time.time()
    total_processed = 0

    # 1. Carregar Checkpoint
    processed_ids = set()
    header_exists = os.path.exists(OUTPUT_CSV_PATH)

    if header_exists:
        try:
            # Carrega apenas a coluna 'record_id' para otimizar a memória
            existing_df = pd.read_csv(OUTPUT_CSV_PATH, usecols=['record_id'])
            processed_ids = set(existing_df['record_id'].astype(str).tolist())
            print(f"💾 Checkpoint encontrado: {len(processed_ids)} registros já processados.")

        except pd.errors.EmptyDataError:
            print("💾 Checkpoint encontrado, mas arquivo está vazio.")
            header_exists = False
        except Exception as e:
            print(f"❌ Erro ao ler checkpoint: {e}. Reiniciando processamento.")
            header_exists = False

    # 2. Processar cada dataset
    for dataset_name, base_path in base_paths.items():
        count, header_exists = process_dataset_simple(
            dataset_name,
            base_path,
            processed_ids,
            OUTPUT_CSV_PATH,
            header_exists
        )
        total_processed += count

    elapsed_time = time.time() - start_time

    # 3. Relatório final
    final_size = len(processed_ids) + total_processed

    print(f"\n🎉 PROCESSAMENTO CONCLUÍDO!")
    print(f"⏱️  Tempo total: {elapsed_time:.2f} segundos")
    print(f"🔄 Novos registros processados nesta execução: {total_processed:,}")
    print(f"📊 Total de registros no arquivo final: {final_size:,}")
    print(f"📁 Arquivo salvo: {OUTPUT_CSV_PATH}")

    # Retorna o DataFrame completo (opcional, só para análise final)
    if final_size > 0:
        return pd.read_csv(OUTPUT_CSV_PATH)
    return pd.DataFrame()

# Executa o script
if __name__ == "__main__":
    print("🚀 Iniciando extração com Checkpointing...")
    df = main_checkpoint()

    if not df.empty:
        # Estatísticas (Opcional: Re-executar o bloco de estatísticas aqui se desejar)
        print(f"\n📋 AMOSTRA DOS DADOS FINAIS:")
        print(df.tail().to_string(index=False))

🚀 Iniciando extração com Checkpointing...
🔍 Processando CODE-15%...
   📁 Total encontrados: 226920. Ignorando 0 já processados.


CODE-15%: 100%|██████████| 226920/226920 [2:41:56<00:00, 23.35it/s]


✅ CODE-15%: 226920 novos registros processados

🎉 PROCESSAMENTO CONCLUÍDO!
⏱️  Tempo total: 10468.73 segundos
🔄 Novos registros processados nesta execução: 226,920
📊 Total de registros no arquivo final: 226,920
📁 Arquivo salvo: /content/drive/MyDrive/Doutorado-Material/Dataset/code15_chagas_dataset.csv

📋 AMOSTRA DOS DADOS FINAIS:
 record_id  dataset  age sex  chagas_label                                                                                               file_path
   1946040 CODE-15%   68   M             0 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part17/1946040.hea
   1848859 CODE-15%   37   M             0 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part17/1848859.hea
   1929150 CODE-15%   56   F             0 /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part17/1929150.hea
   1858808 CODE-15%   22   F             0 /content/drive/MyDrive/Doutorado-Mat

In [ ]:
import pandas as pd

# Caminhos dos arquivos CSV de entrada
csv1_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/code15_chagas_dataset.csv'
csv2_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/chagas_ptbxl_samitrop_dataset.csv'

# Caminho do arquivo de saída
csv_output_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/dataset_chagas.csv'

# 1. Ler os dois CSVs
df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# 2. Concatenar os dois DataFrames
df_final = pd.concat([df1, df2], ignore_index=True)

# 3. Criar a nova coluna "dat_path"
# Supondo que existe a coluna 'file_path' e termina com ".hea"
df_final['dat_path'] = df_final['file_path'].str.replace('.hea', '.dat', regex=False)

# 4. Salvar o resultado em um novo CSV
df_final.to_csv(csv_output_path, index=False)

print(f'Arquivo combinado criado com sucesso em: {csv_output_path}')


Arquivo combinado criado com sucesso em: /content/drive/MyDrive/Doutorado-Material/Dataset/dataset_chagas.csv


In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# Caminho do CSV de entrada (que contém a coluna 'dat_path')
csv_input_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/dataset_chagas.csv'

# Caminho do CSV de saída (onde serão salvos os paths inexistentes)
csv_missing_output_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/arquivos_nao_encontrados.csv'

# 1. Ler o CSV
df = pd.read_csv(csv_input_path)

# 2. Adicionar barra de progresso na verificação
tqdm.pandas(desc="Verificando existência dos arquivos .dat")
df['exists'] = df['dat_path'].progress_apply(lambda x: os.path.exists(x))

# 3. Filtrar os que não existem
df_missing = df[df['exists'] == False]

# 4. Salvar as linhas com arquivos ausentes em um novo CSV
if not df_missing.empty:
    df_missing.to_csv(csv_missing_output_path, index=False)
    print(f'\nForam encontrados {len(df_missing)} arquivos ausentes.')
    print(f'Resultado salvo em: {csv_missing_output_path}')
else:
    print('\n✅ Todos os caminhos em dat_path existem!')


/tmp/ipython-input-3330460649.py:12: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_input_path)
Verificando existência dos arquivos .dat: 100%|██████████| 249534/249534 [10:32<00:00, 394.33it/s]



Foram encontrados 2267 arquivos ausentes.
Resultado salvo em: /content/drive/MyDrive/Doutorado-Material/Dataset/arquivos_nao_encontrados.csv



• Estratégias de aumento de dados, como a criação de mais amostras a partir da segmentação de subespaços do sinal que contém anomalias.**negrito**

# 4 - Divisão do Dataset

In [ ]:
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split, StratifiedKFold
import os

# ================================
# 🔧 CONFIGURAÇÕES
# ================================
csv_input_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/dataset_chagas.csv'
output_dir = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final'
os.makedirs(output_dir, exist_ok=True)

# ================================
# 📥 1. Leitura e preparação
# ================================
df = pd.read_csv(csv_input_path, dtype=str, low_memory=False)

# Garantir que a coluna de label está em formato numérico
df['chagas_label'] = df['chagas_label'].astype(int)

print("Distribuição original das classes:")
print(df['chagas_label'].value_counts(), "\n")

# ================================
# ⚖️ 2. Balanceamento
# ================================
df_majority = df[df['chagas_label'] == 0]
df_minority = df[df['chagas_label'] == 1]

# --- Undersampling ---
df_majority_down = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)
df_under = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42).reset_index(drop=True)

# --- Oversampling ---
df_minority_up = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)
df_over = pd.concat([df_majority, df_minority_up]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanceamento concluído!")
print("Undersampling:", df_under['chagas_label'].value_counts().to_dict())
print("Oversampling:", df_over['chagas_label'].value_counts().to_dict(), "\n")

# ================================
# ✂️ 3. Split: Train / Val / Test (70 / 15 / 15)
# ================================
def split_dataset(df):
    train, temp = train_test_split(
        df,
        test_size=0.3,
        stratify=df['chagas_label'],
        random_state=42
    )
    val, test = train_test_split(
        temp,
        test_size=0.5,
        stratify=temp['chagas_label'],
        random_state=42
    )

    train['set'] = 'train'
    val['set'] = 'val'
    test['set'] = 'test'

    df_split = pd.concat([train, val, test]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_split

df_under_split = split_dataset(df_under)
df_over_split = split_dataset(df_over)

# Salvar as versões com split
df_under_split.to_csv(os.path.join(output_dir, 'dataset_chagas_undersampling_split.csv'), index=False)
df_over_split.to_csv(os.path.join(output_dir, 'dataset_chagas_oversampling_split.csv'), index=False)

print("✅ Versões com divisão train/val/test salvas!\n")

# ================================
# 🔁 4. Cross-validation (5 folds)
# ================================
def create_folds(df, n_splits=5):
    df = df.reset_index(drop=True)  # 🔧 Evita KeyError
    df['fold'] = -1
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (_, val_idx) in enumerate(skf.split(df, df['chagas_label'])):
        df.loc[val_idx, 'fold'] = fold + 1  # Folds numerados de 1 a 5

    return df

df_under_folds = create_folds(df_under.copy())
df_over_folds = create_folds(df_over.copy())

# Salvar as versões com folds
df_under_folds.to_csv(os.path.join(output_dir, 'dataset_chagas_undersampling_folds.csv'), index=False)
df_over_folds.to_csv(os.path.join(output_dir, 'dataset_chagas_oversampling_folds.csv'), index=False)

print("✅ Versões com 5-Fold Cross-Validation salvas!\n")

# ================================
# 📊 5. Resumo final
# ================================
print("Arquivos gerados em:", output_dir)
print("\nArquivos:")
for f in os.listdir(output_dir):
    print(" -", f)

print("\nResumo final:")
print("Undersampling:", df_under['chagas_label'].value_counts().to_dict())
print("Oversampling:", df_over['chagas_label'].value_counts().to_dict())


Distribuição original das classes:
chagas_label
0    244360
1      5174
Name: count, dtype: int64 

Balanceamento concluído!
Undersampling: {1: 5174, 0: 5174}
Oversampling: {1: 244360, 0: 244360} 

✅ Versões com divisão train/val/test salvas!

✅ Versões com 5-Fold Cross-Validation salvas!

Arquivos gerados em: /content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final

Arquivos:
 - dataset_chagas_undersampling_split.csv
 - dataset_chagas_oversampling_split.csv
 - dataset_chagas_undersampling_folds.csv
 - dataset_chagas_oversampling_folds.csv

Resumo final:
Undersampling: {1: 5174, 0: 5174}
Oversampling: {1: 244360, 0: 244360}


# 5 -Verificando a Integridade dos Dados

In [ ]:
# ===========================================
# 1️⃣ Importações e configurações iniciais
# ===========================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import wfdb
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import resample

# ===========================================
# 2️⃣ Funções de carregamento e pré-processamento ECG (CORRIGIDAS)
# ===========================================

def read_ecg_record(dat_path):
    """
    Lê o sinal ECG de 12 derivações do arquivo .dat usando WFDB e normaliza entre 0 e 1.
    Verifica se os arquivos .dat e .hea existem.
    """
    try:
        # Converte bytes para string se necessário
        if isinstance(dat_path, bytes):
            dat_path = dat_path.decode('utf-8')

        # Remove possíveis caracteres problemáticos
        dat_path = dat_path.replace("b'", "").replace("'", "").strip()

        # Verifica se o arquivo .dat existe
        if not os.path.exists(dat_path):
            print(f"Arquivo .dat não encontrado: {dat_path}")
            return np.zeros((5000, 12), dtype=np.float32)

        # Verifica se o arquivo .hea existe
        hea_path = dat_path.replace('.dat', '.hea')
        if not os.path.exists(hea_path):
            print(f"Arquivo .hea não encontrado: {hea_path}")
            return np.zeros((5000, 12), dtype=np.float32)

        # Lê o registro
        record = wfdb.rdrecord(dat_path.replace('.dat', ''))  # Remove extensão para WFDB
        signal = record.p_signal.astype(np.float32)

        # Verifica se o sinal tem o formato esperado
        if signal.shape[1] != 12:
            print(f"Aviso: Sinal em {dat_path} tem {signal.shape[1]} derivações, esperadas 12")
            # Se tiver menos derivações, faz padding
            if signal.shape[1] < 12:
                padding = np.zeros((signal.shape[0], 12 - signal.shape[1]), dtype=np.float32)
                signal = np.hstack([signal, padding])
            # Se tiver mais, trunca
            else:
                signal = signal[:, :12]

        # Normalização min-max para cada derivação (coluna)
        min_val = np.min(signal, axis=0)
        max_val = np.max(signal, axis=0)
        signal_norm = (signal - min_val) / (max_val - min_val + 1e-8)

        # Substitui NaN por 0
        signal_norm = np.nan_to_num(signal_norm)

        return signal_norm
    except Exception as e:
        print(f"Erro ao ler {dat_path}: {e}")
        return np.zeros((5000, 12), dtype=np.float32)

def check_file_exists(dat_path):
    """
    Verifica se ambos os arquivos .dat e .hea existem
    """
    if isinstance(dat_path, bytes):
        dat_path = dat_path.decode('utf-8')

    dat_path = dat_path.replace("b'", "").replace("'", "").strip()

    dat_exists = os.path.exists(dat_path)
    hea_path = dat_path.replace('.dat', '.hea')
    hea_exists = os.path.exists(hea_path)

    return dat_exists and hea_exists

# ===========================================
# 3️⃣ Criação do DataLoader para redes Transformer 1D
# ===========================================

def base_dataset(df, batch_size=32, shuffle=True):
    """
    Cria o dataset do TensorFlow a partir de um DataFrame contendo 'dat_path' e 'chagas_label'.
    """
    # Primeiro, verifica quais arquivos existem
    print("Verificando existência dos arquivos...")
    valid_files = []
    valid_labels = []

    for idx, row in df.iterrows():
        if check_file_exists(row['dat_path']):
            valid_files.append(row['dat_path'])
            valid_labels.append(row['chagas_label'])
        else:
            print(f"Arquivo não encontrado: {row['dat_path']}")

    print(f"Arquivos válidos: {len(valid_files)}/{len(df)}")

    paths = np.array(valid_files)
    labels = np.array(valid_labels)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    def _load_and_preprocess(path, label):
        # Carrega o sinal ECG
        signal = tf.numpy_function(read_ecg_record, [path], tf.float32)

        # Garante o shape mínimo
        signal = tf.ensure_shape(signal, [None, 12])

        # Padding ou truncamento para tamanho fixo (5000 amostras)
        current_length = tf.shape(signal)[0]

        def pad_signal():
            return tf.pad(signal, [[0, 5000 - current_length], [0, 0]])

        def truncate_signal():
            return signal[:5000, :]

        signal = tf.cond(
            current_length < 5000,
            pad_signal,
            truncate_signal
        )

        # Garante shape fixo para o modelo
        signal.set_shape([5000, 12])

        return signal, tf.one_hot(label, depth=2)

    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# ===========================================
# 4️⃣ Função para carregar datasets splitados
# ===========================================

def load_split_datasets(csv_path, batch_size=32):
    """
    Carrega datasets (train, val, test) a partir do CSV gerado na etapa de split.
    """
    # Carrega o DataFrame
    df = pd.read_csv(csv_path)

    # Verifica se as colunas necessárias existem
    required_columns = ['dat_path', 'chagas_label', 'set']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada no CSV")

    # Filtra os datasets
    train_df = df[df['set'] == 'train']
    val_df = df[df['set'] == 'val']
    test_df = df[df['set'] == 'test']

    print(f"Dataset sizes - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # Cria os datasets
    train_ds = base_dataset(train_df, batch_size, shuffle=True)
    val_ds = base_dataset(val_df, batch_size, shuffle=False)
    test_ds = base_dataset(test_df, batch_size, shuffle=False)

    return train_ds, val_ds, test_ds

# ===========================================
# 5️⃣ Função para verificar integridade do dataset
# ===========================================

def verify_dataset_integrity(csv_path):
    """
    Verifica a integridade de todos os arquivos no dataset
    """
    df = pd.read_csv(csv_path)

    missing_files = []
    valid_files = []

    print("Verificando integridade do dataset...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        dat_path = row['dat_path']

        if isinstance(dat_path, bytes):
            dat_path = dat_path.decode('utf-8')
        dat_path = dat_path.replace("b'", "").replace("'", "").strip()

        # Verifica arquivo .dat
        if not os.path.exists(dat_path):
            missing_files.append(('dat', dat_path))
            continue

        # Verifica arquivo .hea
        hea_path = dat_path.replace('.dat', '.hea')
        if not os.path.exists(hea_path):
            missing_files.append(('hea', hea_path))
            continue

        valid_files.append(dat_path)

    print(f"\n=== RESULTADO DA VERIFICAÇÃO ===")
    print(f"Total de registros no CSV: {len(df)}")
    print(f"Arquivos válidos: {len(valid_files)}")
    print(f"Arquivos faltantes: {len(missing_files)}")

    if missing_files:
        print("\nArquivos faltantes (primeiros 10):")
        for file_type, path in missing_files[:10]:
            print(f"  {file_type.upper()}: {path}")

    return valid_files, missing_files

# ===========================================
# 6️⃣ Função para criar dataset apenas com arquivos válidos
# ===========================================

def create_filtered_dataset(csv_path, output_csv_path=None):
    """
    Cria um novo CSV apenas com os arquivos que existem
    """
    df = pd.read_csv(csv_path)

    valid_indices = []

    print("Filtrando dataset...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        if check_file_exists(row['dat_path']):
            valid_indices.append(idx)

    filtered_df = df.iloc[valid_indices].copy()

    print(f"Dataset original: {len(df)} registros")
    print(f"Dataset filtrado: {len(filtered_df)} registros")

    if output_csv_path:
        filtered_df.to_csv(output_csv_path, index=False)
        print(f"Dataset filtrado salvo em: {output_csv_path}")

    return filtered_df

# ===========================================
# 7️⃣ Funções auxiliares para debug
# ===========================================

def debug_single_file(dat_path):
    """
    Função para debug de um único arquivo .dat
    """
    print(f"Debug do arquivo: {dat_path}")

    # Testa a leitura direta
    try:
        if isinstance(dat_path, bytes):
            dat_path_str = dat_path.decode('utf-8')
        else:
            dat_path_str = str(dat_path)

        dat_path_str = dat_path_str.replace("b'", "").replace("'", "").strip()
        print(f"Path processado: {dat_path_str}")

        # Verifica se os arquivos existem
        dat_exists = os.path.exists(dat_path_str)
        hea_path = dat_path_str.replace('.dat', '.hea')
        hea_exists = os.path.exists(hea_path)

        print(f"Arquivo .dat existe: {dat_exists}")
        print(f"Arquivo .hea existe: {hea_exists}")

        if not dat_exists:
            print(f"❌ Arquivo .dat não encontrado: {dat_path_str}")
            return False

        if not hea_exists:
            print(f"❌ Arquivo .hea não encontrado: {hea_path}")
            return False

        # Tenta ler com WFDB
        record = wfdb.rdrecord(dat_path_str.replace('.dat', ''))  # Remove extensão
        signal = record.p_signal
        print(f"✅ Leitura bem-sucedida!")
        print(f"Shape do sinal: {signal.shape}")
        print(f"Tipo do sinal: {signal.dtype}")
        print(f"Valores min/max: {np.min(signal):.4f} / {np.max(signal):.4f}")

        return True
    except Exception as e:
        print(f"❌ ERRO: {e}")
        return False

# ===========================================
# 8️⃣ Exemplo de uso
# ===========================================

if __name__ == "__main__":
    # Caminho para o CSV
    csv_path = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split.csv'

    try:
        # 1. Primeiro verifica a integridade
        print("=== VERIFICAÇÃO DE INTEGRIDADE ===")
        valid_files, missing_files = verify_dataset_integrity(csv_path)

        if missing_files:
            print(f"\n⚠️  Encontrados {len(missing_files)} arquivos faltantes!")
            print("Criando dataset filtrado...")

            # Cria dataset filtrado
            filtered_csv_path = csv_path.replace('.csv', '_filtered.csv')
            filtered_df = create_filtered_dataset(csv_path, filtered_csv_path)

            # Usa o dataset filtrado
            csv_path = filtered_csv_path

        # 2. Carrega os datasets
        print("\n=== CARREGANDO DATASETS ===")
        train_ds, val_ds, test_ds = load_split_datasets(csv_path, batch_size=64)

        # 3. Testa um batch
        print("\n=== TESTANDO DATASET ===")
        for batch in train_ds.take(1):
            x, y = batch
            print("✅ Forma do batch ECG:", x.shape)
            print("✅ Forma do batch labels:", y.shape)
            print("✅ Valores mínimos do ECG:", tf.reduce_min(x, axis=[0, 1]).numpy())
            print("✅ Valores máximos do ECG:", tf.reduce_max(x, axis=[0, 1]).numpy())
            print("✅ Distribuição dos labels:", tf.reduce_sum(y, axis=0).numpy())

    except Exception as e:
        print(f"❌ Erro ao carregar datasets: {e}")
        import traceback
        traceback.print_exc()

# ===========================================
# 9️⃣ Teste de arquivo específico
# ===========================================

# Teste o arquivo problemático
print("\n=== DEBUG DO ARQUIVO PROBLEMÁTICO ===")
debug_single_file(b'/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part0/113.dat')

=== VERIFICAÇÃO DE INTEGRIDADE ===
Verificando integridade do dataset...


100%|██████████| 488720/488720 [18:20<00:00, 444.26it/s] 



=== RESULTADO DA VERIFICAÇÃO ===
Total de registros no CSV: 488720
Arquivos válidos: 484389
Arquivos faltantes: 4331

Arquivos faltantes (primeiros 10):
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part4/419648.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part2/2855507.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part15/1270930.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part16/2068305.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part16/1843976.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part16/1990091.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part16/1926115.dat
  DAT: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/

100%|██████████| 488720/488720 [07:12<00:00, 1129.21it/s]


Dataset original: 488720 registros
Dataset filtrado: 484392 registros
Dataset filtrado salvo em: /content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split_filtered.csv

=== CARREGANDO DATASETS ===
Dataset sizes - Train: 339064, Val: 72705, Test: 72623
Verificando existência dos arquivos...
Arquivos válidos: 339064/339064
Verificando existência dos arquivos...
Arquivos válidos: 72705/72705
Verificando existência dos arquivos...
Arquivos válidos: 72623/72623

=== TESTANDO DATASET ===
✅ Forma do batch ECG: (64, 5000, 12)
✅ Forma do batch labels: (64, 2)
✅ Valores mínimos do ECG: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
✅ Valores máximos do ECG: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
✅ Distribuição dos labels: [26. 38.]

=== DEBUG DO ARQUIVO PROBLEMÁTICO ===
Debug do arquivo: b'/content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Chagas/code15_output/exams_part0/113.dat'
Path processado: /content/drive/MyDrive/Doutorado-Material/Dataset/Dataset-Cha

True

# 6 -Modelo Transformers de Teste

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ===========================================
# 1️⃣ Definição das Camadas do Transformer (CORRIGIDO)
# ===========================================

class StochasticDepth(layers.Layer):
    """Implementação do Stochastic Depth corrigida para Graph mode"""
    def __init__(self, survival_probability, **kwargs):
        super().__init__(**kwargs)
        self.survival_probability = survival_probability

    def call(self, inputs, training=None):
        if training is None:
            training = False

        # Durante inferência, usa apenas o novo caminho
        if not training:
            return inputs[1]

        # Durante treinamento, aplica stochastic depth usando operações do TF
        batch_size = tf.shape(inputs[0])[0]
        random_tensor = tf.random.uniform([batch_size], 0, 1)

        # Criar máscara binária
        binary_mask = tf.cast(random_tensor < self.survival_probability, tf.float32)

        # Aplicar máscara
        binary_mask = tf.reshape(binary_mask, [-1, 1, 1])  # Para broadcasting

        output = (
            inputs[1] * (binary_mask / self.survival_probability) +
            inputs[0] * (1.0 - binary_mask)
        )

        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "survival_probability": self.survival_probability,
        })
        return config

class ViTEmbeddings(layers.Layer):
    def __init__(self, patch_size, hidden_size, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.hidden_size = hidden_size

        self.patch_embeddings = layers.Conv1D(
            filters=hidden_size,
            kernel_size=patch_size,
            strides=patch_size,
            name="patch_embeddings"
        )
        self.dropout = layers.Dropout(rate=dropout)

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            shape=(1, 1, self.hidden_size),
            trainable=True,
            name="cls_token"
        )

        num_patches = input_shape[1] // self.patch_size
        self.position_embeddings = self.add_weight(
            shape=(1, num_patches + 1, self.hidden_size),
            trainable=True,
            name="position_embeddings"
        )
        super().build(input_shape)

    def call(self, inputs: tf.Tensor, training: bool = False) -> tf.Tensor:
        inputs_shape = tf.shape(inputs)  # N, 5000, 12

        # Aplica convolução para criar patches
        embeddings = self.patch_embeddings(inputs, training=training)

        # Adiciona o token [CLS] aos patches
        cls_tokens = tf.repeat(self.cls_token, repeats=inputs_shape[0], axis=0)
        embeddings = tf.concat((cls_tokens, embeddings), axis=1)

        # Adiciona embeddings posicionais
        embeddings = embeddings + self.position_embeddings
        embeddings = self.dropout(embeddings, training=training)

        return embeddings

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
            "hidden_size": self.hidden_size,
        })
        return config

class MLP(layers.Layer):
    def __init__(self, mlp_dim, out_dim=None, activation="gelu", dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.mlp_dim = mlp_dim
        self.out_dim = out_dim
        self.activation = activation
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.dense1 = layers.Dense(self.mlp_dim, name="dense1")
        self.activation1 = layers.Activation(self.activation)
        self.dropout = layers.Dropout(self.dropout_rate)
        self.dense2 = layers.Dense(
            input_shape[-1] if self.out_dim is None else self.out_dim,
            name="dense2"
        )
        super().build(input_shape)

    def call(self, inputs: tf.Tensor, training: bool = False):
        x = self.dense1(inputs)
        x = self.activation1(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        x = self.dropout(x, training=training)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "mlp_dim": self.mlp_dim,
            "out_dim": self.out_dim,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

class Block(layers.Layer):
    def __init__(
        self,
        num_heads,
        attention_dim,
        mlp_dim,
        attention_dropout=0.0,
        sd_survival_probability=1.0,
        activation="gelu",
        dropout=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.attention_dim = attention_dim
        self.mlp_dim = mlp_dim
        self.attention_dropout = attention_dropout
        self.sd_survival_probability = sd_survival_probability
        self.activation = activation
        self.dropout = dropout

        self.norm_before = layers.LayerNormalization(name="norm_before")
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=attention_dim // num_heads,
            dropout=attention_dropout,
            name="attention"
        )
        self.stochastic_depth1 = StochasticDepth(sd_survival_probability)
        self.stochastic_depth2 = StochasticDepth(sd_survival_probability)
        self.norm_after = layers.LayerNormalization(name="norm_after")
        self.mlp = MLP(
            mlp_dim=mlp_dim,
            activation=activation,
            dropout=dropout,
            name="mlp"
        )

    def call(self, inputs, training=False):
        # Primeira sub-camada: Attention + Skip Connection
        x_norm = self.norm_before(inputs, training=training)
        attn_output = self.attn(x_norm, x_norm, training=training)
        x = self.stochastic_depth1([inputs, attn_output], training=training)

        # Segunda sub-camada: MLP + Skip Connection
        x_norm2 = self.norm_after(x, training=training)
        mlp_output = self.mlp(x_norm2, training=training)
        output = self.stochastic_depth2([x, mlp_output], training=training)

        return output

    def get_attention_scores(self, inputs):
        x = self.norm_before(inputs, training=False)
        _, weights = self.attn(x, x, training=False, return_attention_scores=True)
        return weights

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_heads": self.num_heads,
            "attention_dim": self.attention_dim,
            "mlp_dim": self.mlp_dim,
            "attention_dropout": self.attention_dropout,
            "sd_survival_probability": self.sd_survival_probability,
            "activation": self.activation,
            "dropout": self.dropout,
        })
        return config

class VisionTransformer(tf.keras.Model):
    def __init__(
        self,
        patch_size=20,
        hidden_size=768,
        depth=6,
        num_heads=6,
        mlp_dim=256,
        num_classes=2,
        dropout=0.1,
        sd_survival_probability=0.9,
        attention_dropout=0.0,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.patch_size = patch_size
        self.hidden_size = hidden_size
        self.depth = depth
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.num_classes = num_classes

        # Camada de embeddings
        self.embeddings = ViTEmbeddings(
            patch_size=patch_size,
            hidden_size=hidden_size,
            dropout=dropout,
            name="embeddings"
        )

        # Blocos Transformer
        sd_probs = tf.linspace(1.0, sd_survival_probability, depth)
        self.blocks = [
            Block(
                num_heads=num_heads,
                attention_dim=hidden_size,
                mlp_dim=mlp_dim,
                attention_dropout=attention_dropout,
                sd_survival_probability=sd_probs[i].numpy().item(),
                dropout=dropout,
                name=f"block_{i}"
            )
            for i in range(depth)
        ]

        # Camadas finais
        self.norm = layers.LayerNormalization(name="final_norm")
        self.head = layers.Dense(num_classes, name="classification_head")
        self.softmax = layers.Activation('softmax', dtype='float32', name='predictions')

    def call(self, inputs: tf.Tensor, training: bool = False) -> tf.Tensor:
        # Embeddings
        x = self.embeddings(inputs, training=training)

        # Blocos Transformer
        for block in self.blocks:
            x = block(x, training=training)

        # Classificação
        x = self.norm(x, training=training)
        x = x[:, 0]  # Token [CLS] para classificação
        logits = self.head(x)
        return self.softmax(logits)

    def get_last_selfattention(self, inputs: tf.Tensor):
        x = self.embeddings(inputs, training=False)
        for block in self.blocks[:-1]:
            x = block(x, training=False)
        return self.blocks[-1].get_attention_scores(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
            "hidden_size": self.hidden_size,
            "depth": self.depth,
            "num_heads": self.num_heads,
            "mlp_dim": self.mlp_dim,
            "num_classes": self.num_classes,
        })
        return config

# ===========================================
# 2️⃣ Função para Criar e Compilar o Modelo
# ===========================================

def create_vit_model(
    input_shape=(5000, 12),
    patch_size=20,
    hidden_size=768,
    depth=6,
    num_heads=6,
    mlp_dim=256,
    num_classes=2,
    dropout=0.1,
    learning_rate=0.0001
):
    """
    Cria e compila o modelo Vision Transformer para classificação de ECG
    """
    # Criar modelo
    vit = VisionTransformer(
        patch_size=patch_size,
        hidden_size=hidden_size,
        depth=depth,
        num_heads=num_heads,
        mlp_dim=mlp_dim,
        num_classes=num_classes,
        dropout=dropout,
        sd_survival_probability=0.9,
    )

    # Compilar modelo
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss = tf.keras.losses.CategoricalCrossentropy()
    metrics = [
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]

    vit.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    return vit

# ===========================================
# 3️⃣ Versão Simplificada do Modelo
# ===========================================

def create_simple_vit_model(
    input_shape=(5000, 12),
    patch_size=20,
    hidden_size=256,
    depth=4,
    num_heads=4,
    mlp_dim=128,
    num_classes=2,
    dropout=0.1,
    learning_rate=0.0001
):
    """
    Versão simplificada do ViT para treinamento mais rápido
    """
    return create_vit_model(
        input_shape=input_shape,
        patch_size=patch_size,
        hidden_size=hidden_size,
        depth=depth,
        num_heads=num_heads,
        mlp_dim=mlp_dim,
        num_classes=num_classes,
        dropout=dropout,
        learning_rate=learning_rate
    )

# ===========================================
# 4️⃣ Versão SEM Stochastic Depth (Alternativa)
# ===========================================

def create_vit_without_sd(
    input_shape=(5000, 12),
    patch_size=20,
    hidden_size=256,
    depth=4,
    num_heads=4,
    mlp_dim=128,
    num_classes=2,
    dropout=0.1,
    learning_rate=0.0001
):
    """
    Versão do ViT sem Stochastic Depth para máxima estabilidade
    """
    class SimpleBlock(layers.Layer):
        def __init__(self, num_heads, attention_dim, mlp_dim, dropout=0.0, **kwargs):
            super().__init__(**kwargs)
            self.norm1 = layers.LayerNormalization()
            self.attn = layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=attention_dim // num_heads,
                dropout=dropout,
            )
            self.norm2 = layers.LayerNormalization()
            self.mlp = MLP(mlp_dim=mlp_dim, dropout=dropout)

        def call(self, inputs, training=False):
            # Attention com skip connection
            x = self.norm1(inputs, training=training)
            attn_output = self.attn(x, x, training=training)
            x = inputs + attn_output

            # MLP com skip connection
            x2 = self.norm2(x, training=training)
            mlp_output = self.mlp(x2, training=training)
            return x + mlp_output

    # Criar modelo simples
    inputs = layers.Input(shape=input_shape)

    # Embeddings
    embeddings = ViTEmbeddings(patch_size, hidden_size, dropout)(inputs)

    # Blocos Transformer
    x = embeddings
    for i in range(depth):
        x = SimpleBlock(
            num_heads=num_heads,
            attention_dim=hidden_size,
            mlp_dim=mlp_dim,
            dropout=dropout,
            name=f"block_{i}"
        )(x)

    # Classificação
    x = layers.LayerNormalization()(x)
    x = x[:, 0]  # Token CLS
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Compilar
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy', 'auc', 'precision', 'recall']
    )

    return model

# ===========================================
# 5️⃣ Exemplo de Uso Completo
# ===========================================

if __name__ == "__main__":
    # Configurações
    BATCH_SIZE = 32
    EPOCHS = 5
    LEARNING_RATE = 0.0001

    try:
        # 1. Criar dados dummy para teste
        print("=== CRIANDO DADOS DUMMY PARA TESTE ===")
        dummy_x = np.random.normal(0, 1, (100, 5000, 12)).astype(np.float32)
        dummy_y = tf.keras.utils.to_categorical(np.random.randint(0, 2, 100), 2)

        train_ds = tf.data.Dataset.from_tensor_slices((dummy_x[:80], dummy_y[:80])).batch(BATCH_SIZE)
        val_ds = tf.data.Dataset.from_tensor_slices((dummy_x[80:90], dummy_y[80:90])).batch(BATCH_SIZE)
        test_ds = tf.data.Dataset.from_tensor_slices((dummy_x[90:], dummy_y[90:])).batch(BATCH_SIZE)

        # 2. Verificar um batch
        print("\n=== VERIFICANDO BATCH ===")
        for batch in train_ds.take(1):
            x, y = batch
            print(f"Shape do ECG: {x.shape}")
            print(f"Shape dos labels: {y.shape}")

        # 3. Criar modelo SEM Stochastic Depth para máxima estabilidade
        print("\n=== CRIANDO MODELO VISION TRANSFORMER SIMPLIFICADO ===")
        vit_model = create_vit_without_sd(
            input_shape=(5000, 12),
            patch_size=20,
            hidden_size=256,
            depth=4,
            num_heads=4,
            mlp_dim=128,
            num_classes=2,
            learning_rate=LEARNING_RATE
        )

        # Resumo do modelo
        print("\n=== RESUMO DO MODELO ===")
        vit_model.summary()

        # 4. Treinar modelo
        print("\n=== INICIANDO TREINAMENTO DE TESTE ===")

        history = vit_model.fit(
            train_ds,
            epochs=EPOCHS,
            validation_data=val_ds,
            verbose=1
        )

        # 5. Avaliar modelo
        print("\n=== AVALIANDO MODELO ===")
        test_results = vit_model.evaluate(test_ds, verbose=1)
        print(f"Resultados no teste:")
        print(f"  Loss: {test_results[0]:.4f}")
        print(f"  Accuracy: {test_results[1]:.4f}")
        print(f"  AUC: {test_results[2]:.4f}")
        print(f"  Precision: {test_results[3]:.4f}")
        print(f"  Recall: {test_results[4]:.4f}")

        print("✅ Modelo Vision Transformer criado e testado com sucesso!")

        # 6. Testar predição
        print("\n=== TESTANDO PREDIÇÃO ===")
        sample_batch = next(iter(test_ds))
        predictions = vit_model.predict(sample_batch[0][:2], verbose=0)
        print(f"Predictions shape: {predictions.shape}")
        print(f"Primeiras predições: {predictions[0]}")

    except Exception as e:
        print(f"Erro durante execução: {e}")
        import traceback
        traceback.print_exc()

=== CRIANDO DADOS DUMMY PARA TESTE ===

=== VERIFICANDO BATCH ===
Shape do ECG: (32, 5000, 12)
Shape dos labels: (32, 2)

=== CRIANDO MODELO VISION TRANSFORMER SIMPLIFICADO ===

=== RESUMO DO MODELO ===


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 5000, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vi_t_embeddings (ViTEmbeddings) │ (None, 251, 256)       │       126,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_0 (SimpleBlock)           │ (None, 251, 256)       │       330,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1 (SimpleBlock)           │ (None, 251, 256)       │       330,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2 (SimpleBlock)           │ (None, 251, 256)       │       330,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3 (SimpleBlock)           │ (None, 251, 256)       │       330,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_8           │ (None, 251, 256)       │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,447,682 (5.52 MB)

 Trainable params: 1,447,682 (5.52 MB)

 Non-trainable params: 0 (0.00 B)


=== INICIANDO TREINAMENTO DE TESTE ===
Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 32s 4s/step - accuracy: 0.4812 - auc: 0.4728 - loss: 0.9885 - precision: 0.4812 - recall: 0.4812 - val_accuracy: 0.5000 - val_auc: 0.5100 - val_loss: 0.7928 - val_precision: 0.5000 - val_recall: 0.5000
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 3s/step - accuracy: 0.4383 - auc: 0.4407 - loss: 0.8748 - precision: 0.4383 - recall: 0.4383 - val_accuracy: 0.5000 - val_auc: 0.5200 - val_loss: 0.8998 - val_precision: 0.5000 - val_recall: 0.5000
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.6125 - auc: 0.6275 - loss: 0.7020 - precision: 0.6125 - recall: 0.6125 - val_accuracy: 0.5000 - val_auc: 0.5400 - val_loss: 0.7418 - val_precision: 0.5000 - val_recall: 0.5000
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.4617 - auc: 0.4771 - loss: 0.8282 - precision: 0.4617 - recall: 0.4617 - val_accuracy: 0.5000 - val_auc: 0.5200 - val_loss: 0.7137 - val_precision: 0.5000 - val_recall: 0.5000
Epoch 5/5
3/

# 7 -Modelo Transformers Treinamento - V1 - 50 epocas

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import wfdb
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ===========================================
# 0️⃣ CONFIGURAÇÃO GPU E PERFORMANCE
# ===========================================

# Configurações para máxima performance na GPU
def setup_gpu():
    """Configura TensorFlow para máxima performance na GPU"""
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            # Permite crescimento de memória para evitar OOM
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)

            # Configuração de estratégia para multi-GPU se disponível
            if len(gpus) > 1:
                strategy = tf.distribute.MirroredStrategy()
                print(f'🚀 {len(gpus)} GPUs detectadas. Usando estratégia MirroredStrategy')
                return strategy
            else:
                strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
                print('🚀 1 GPU detectada. Configurando para máxima performance')
                return strategy

        except RuntimeError as e:
            print(f'❌ Erro na configuração GPU: {e}')

    print('⚠️  Nenhuma GPU detectada. Usando CPU')
    return tf.distribute.OneDeviceStrategy(device="/cpu:0")

# Configurar GPU no início
strategy = setup_gpu()

# ===========================================
# 1️⃣ FUNÇÕES DE CARREGAMENTO OTIMIZADAS PARA GPU
# ===========================================

def read_ecg_record_py(path):
    """Função Python para leitura ECG"""
    try:
        path = path.decode('utf-8').replace("b'", "").replace("'", "").strip()

        if not os.path.exists(path) or not os.path.exists(path.replace('.dat', '.hea')):
            return np.zeros((5000, 12), dtype=np.float32)

        record = wfdb.rdrecord(path.replace('.dat', ''))
        signal = record.p_signal.astype(np.float32)

        # Garante 12 derivações
        if signal.shape[1] != 12:
            if signal.shape[1] < 12:
                padding = np.zeros((signal.shape[0], 12 - signal.shape[1]), dtype=np.float32)
                signal = np.hstack([signal, padding])
            else:
                signal = signal[:, :12]

        # Garante 5000 amostras
        if signal.shape[0] != 5000:
            if signal.shape[0] < 5000:
                padding = np.zeros((5000 - signal.shape[0], 12), dtype=np.float32)
                signal = np.vstack([signal, padding])
            else:
                signal = signal[:5000, :]

        # Normalização rápida
        min_val = np.min(signal, axis=0)
        max_val = np.max(signal, axis=0)
        range_val = np.maximum(max_val - min_val, 1e-8)  # Evita divisão por zero

        signal_norm = (signal - min_val) / range_val
        return np.nan_to_num(signal_norm, nan=0.0, posinf=1.0, neginf=0.0)

    except Exception as e:
        return np.zeros((5000, 12), dtype=np.float32)

def create_optimized_dataset(df, batch_size=64, shuffle=True, cache=True):
    """Cria dataset otimizado para GPU com pipeline eficiente"""
    paths = df['dat_path'].values
    labels = df['chagas_label'].values

    print(f"📁 Criando dataset com {len(paths)} amostras...")

    # Dataset básico
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    # Shuffle primeiro (mais eficiente)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, len(paths)), reshuffle_each_iteration=True)

    # Mapeamento paralelizado
    def _load_and_preprocess(path, label):
        signal = tf.numpy_function(read_ecg_record_py, [path], tf.float32)
        signal = tf.ensure_shape(signal, [5000, 12])
        return signal, tf.one_hot(label, depth=2)

    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch
    ds = ds.batch(batch_size)

    # Cache se possível (para datasets que cabem na memória)
    if cache:
        ds = ds.cache()

    # Prefetch para sobrepor computação de dados e treinamento
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

def load_split_datasets_gpu(csv_path, batch_size=64):
    """Carrega datasets otimizados para GPU"""
    df = pd.read_csv(csv_path)

    train_df = df[df['set'] == 'train']
    val_df = df[df['set'] == 'val']
    test_df = df[df['set'] == 'test']

    print(f"📊 Dataset sizes - Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

    # Usar cache apenas para treino (val e test não precisam)
    train_ds = create_optimized_dataset(train_df, batch_size, shuffle=True, cache=True)
    val_ds = create_optimized_dataset(val_df, batch_size, shuffle=False, cache=False)
    test_ds = create_optimized_dataset(test_df, batch_size, shuffle=False, cache=False)

    return train_ds, val_ds, test_ds

# ===========================================
# 2️⃣ ARQUITETURA VISION TRANSFORMER CORRIGIDA
# ===========================================

class PatchEmbedding(layers.Layer):
    """Layer para embeddings de patches com CLS token"""
    def __init__(self, patch_size, hidden_size, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.hidden_size = hidden_size
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.patch_embeddings = layers.Conv1D(
            filters=self.hidden_size,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            use_bias=True,
            name="patch_conv"
        )

        # CLS token como weight
        self.cls_token = self.add_weight(
            shape=(1, 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="cls_token"
        )

        # Positional embeddings
        num_patches = input_shape[1] // self.patch_size
        self.position_embeddings = self.add_weight(
            shape=(1, num_patches + 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="position_embeddings"
        )

        self.dropout = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs):
        # Gerar patches
        patches = self.patch_embeddings(inputs)

        # Adicionar CLS token
        batch_size = tf.shape(inputs)[0]
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        embeddings = tf.concat([cls_tokens, patches], axis=1)

        # Adicionar positional embeddings
        embeddings = embeddings + self.position_embeddings

        return self.dropout(embeddings)

class TransformerBlock(layers.Layer):
    """Bloco Transformer otimizado para GPU"""
    def __init__(self, num_heads, hidden_size, mlp_ratio=2, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.hidden_size = hidden_size
        self.mlp_ratio = mlp_ratio
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attention = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.hidden_size // self.num_heads,
            dropout=self.dropout_rate,
            use_bias=True
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        # MLP
        self.mlp_dense1 = layers.Dense(self.hidden_size * self.mlp_ratio, activation='gelu')
        self.mlp_dropout1 = layers.Dropout(self.dropout_rate)
        self.mlp_dense2 = layers.Dense(self.hidden_size)
        self.mlp_dropout2 = layers.Dropout(self.dropout_rate)

        self.dropout = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        # Attention com skip connection
        x_norm = self.norm1(inputs, training=training)
        attn_output = self.attention(x_norm, x_norm, training=training)
        x = inputs + self.dropout(attn_output, training=training)

        # MLP com skip connection
        x_norm2 = self.norm2(x, training=training)
        mlp_output = self.mlp_dense1(x_norm2)
        mlp_output = self.mlp_dropout1(mlp_output, training=training)
        mlp_output = self.mlp_dense2(mlp_output)
        mlp_output = self.mlp_dropout2(mlp_output, training=training)

        return x + mlp_output

def create_vit_model_gpu_optimized(
    input_shape=(5000, 12),
    patch_size=200,
    hidden_size=256,
    depth=4,
    num_heads=4,
    mlp_ratio=2,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """Cria modelo ViT otimizado para GPU - VERSÃO CORRIGIDA"""

    with strategy.scope():
        # Input
        inputs = layers.Input(shape=input_shape, name="ecg_input")

        # Patch Embedding com CLS token
        x = PatchEmbedding(
            patch_size=patch_size,
            hidden_size=hidden_size,
            dropout=dropout,
            name="patch_embedding"
        )(inputs)

        # Transformer Blocks
        for i in range(depth):
            x = TransformerBlock(
                num_heads=num_heads,
                hidden_size=hidden_size,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                name=f"transformer_block_{i}"
            )(x)

        # Classification Head
        x = layers.LayerNormalization(epsilon=1e-6, name="final_norm")(x)

        # Pegar apenas o token CLS (primeira posição)
        cls_output = layers.Lambda(lambda x: x[:, 0], name="cls_token_extractor")(x)

        # MLP Head
        x = layers.Dropout(dropout, name="head_dropout1")(cls_output)
        x = layers.Dense(256, activation='gelu', kernel_initializer='he_normal', name="head_dense1")(x)
        x = layers.Dropout(dropout, name="head_dropout2")(x)
        x = layers.Dense(128, activation='gelu', kernel_initializer='he_normal', name="head_dense2")(x)
        x = layers.Dropout(dropout, name="head_dropout3")(x)

        outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name="vit_gpu_optimized")

        # Otimizador
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=0.01,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-8
        )

        # Compilar
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.AUC(name='auc', curve='ROC'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.F1Score(name='f1_score')
            ]
        )

    return model

# ===========================================
# 3️⃣ DATA AUGMENTATION ACELERADA POR GPU
# ===========================================

class ECGDataAugmentation(layers.Layer):
    """Layer de data augmentation para ECG"""
    def __init__(self, noise_stddev=0.02, scale_range=(0.9, 1.1), **kwargs):
        super().__init__(**kwargs)
        self.noise_stddev = noise_stddev
        self.scale_range = scale_range

    def call(self, inputs, training=False):
        if not training:
            return inputs

        # Ruído gaussiano
        noise = tf.random.normal(
            shape=tf.shape(inputs),
            mean=0.0,
            stddev=self.noise_stddev
        )
        augmented = inputs + noise

        # Variação de escala por lead
        scales = tf.random.uniform(
            [12],
            self.scale_range[0],
            self.scale_range[1]
        )
        augmented = augmented * scales

        # Clip para manter valores válidos
        augmented = tf.clip_by_value(augmented, 0.0, 1.0)

        return augmented

# ===========================================
# 4️⃣ TREINAMENTO OTIMIZADO PARA GPU
# ===========================================

def get_gpu_optimized_callbacks(checkpoint_path, log_dir=None):
    """Callbacks otimizados para treinamento em GPU"""
    callbacks = [
        # Checkpoint com val_auc como métrica principal
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor='val_auc',
            save_best_only=True,
            save_weights_only=False,
            mode='max',
            verbose=1
        ),

        # Early Stopping
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            patience=20,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),

        # Redução de LR
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]

    return callbacks

def train_vit_model_gpu(csv_path, output_dir, epochs=150, batch_size=64):
    """Treinamento otimizado para GPU"""

    # Criar diretórios
    os.makedirs(output_dir, exist_ok=True)
    checkpoint_path = f"{output_dir}/best_model.h5"

    print("🚀 INICIANDO CARREGAMENTO DE DADOS OTIMIZADO PARA GPU")
    print("=" * 60)

    # Carregar datasets otimizados
    train_ds, val_ds, test_ds = load_split_datasets_gpu(csv_path, batch_size=batch_size)

    # Class weights
    print("⚖️ CALCULANDO CLASS WEIGHTS...")
    df = pd.read_csv(csv_path)
    train_df = df[df['set'] == 'train']
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['chagas_label']),
        y=train_df['chagas_label']
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    print(f"Class weights: {class_weight_dict}")

    # Criar modelo otimizado para GPU
    print("🧠 CRIANDO MODELO VISION TRANSFORMER OTIMIZADO PARA GPU...")
    model = create_vit_model_gpu_optimized(
        hidden_size=256,
        depth=4,
        num_heads=4,
        mlp_ratio=2,
        dropout=0.2,
        learning_rate=0.001
    )

    # Callbacks otimizados
    callbacks = get_gpu_optimized_callbacks(checkpoint_path, output_dir)

    print("📈 RESUMO DO MODELO:")
    model.summary()

    # Treinamento
    print("🎯 INICIANDO TREINAMENTO ACELERADO POR GPU...")
    print("=" * 60)

    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    # Carregar melhor modelo
    print("📥 CARREGANDO MELHOR MODELO...")
    best_model = tf.keras.models.load_model(checkpoint_path)

    # Avaliação final
    print("📊 AVALIAÇÃO FINAL...")
    test_results = best_model.evaluate(test_ds, verbose=1, return_dict=True)

    print("\n🎉 RESULTADOS FINAIS:")
    print("=" * 40)
    for metric, value in test_results.items():
        print(f"{metric:12}: {value:.4f}")

    # Salvar modelo final
    final_path = f"{output_dir}/final_model.h5"
    best_model.save(final_path)
    print(f"\n💾 Modelo final salvo em: {final_path}")

    return best_model, history, test_results

# ===========================================
# 5️⃣ ANÁLISE DE PERFORMANCE ACELERADA
# ===========================================

def analyze_model_performance_gpu(model, test_ds, batch_size=64):
    """Análise de performance otimizada para GPU"""
    print("🔍 ANALISANDO PERFORMANCE DO MODELO...")

    # Predição em batch para GPU
    y_pred_proba = model.predict(test_ds, batch_size=batch_size, verbose=1)

    # Coletar labels verdadeiros
    y_true = np.concatenate([y for x, y in test_ds], axis=0)
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_pred_proba, axis=1)

    print("\n📋 RELATÓRIO DE CLASSIFICAÇÃO:")
    print("=" * 50)
    print(classification_report(y_true_labels, y_pred, target_names=['Não Chagas', 'Chagas']))

    print("\n🎯 MATRIZ DE CONFUSÃO:")
    print("=" * 30)
    cm = confusion_matrix(y_true_labels, y_pred)
    print(cm)

    auc_score = roc_auc_score(y_true_labels, y_pred_proba[:, 1])
    print(f"\n📊 AUC Score: {auc_score:.4f}")

    return y_true_labels, y_pred, y_pred_proba

# ===========================================
# 6️⃣ SCRIPT PRINCIPAL OTIMIZADO
# ===========================================

if __name__ == "__main__":
    # Configurações otimizadas para GPU
    CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_undersampling_split.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized'
    EPOCHS = 50
    BATCH_SIZE = 128

    try:
        print("🚀 INICIANDO TREINAMENTO VISION TRANSFORMER OTIMIZADO PARA GPU")
        print("=" * 70)
        print(f"📁 CSV: {CSV_PATH}")
        print(f"📂 Output: {OUTPUT_DIR}")
        print(f"🔄 Epochs: {EPOCHS}")
        print(f"📦 Batch Size: {BATCH_SIZE}")
        print("=" * 70)

        # Treinamento otimizado para GPU
        model, history, test_results = train_vit_model_gpu(
            csv_path=CSV_PATH,
            output_dir=OUTPUT_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE
        )

        # Análise de performance
        print("\n" + "=" * 70)
        print("📊 ANÁLISE DETALHADA DE PERFORMANCE")
        print("=" * 70)

        _, _, test_ds = load_split_datasets_gpu(CSV_PATH, batch_size=BATCH_SIZE)
        y_true, y_pred, y_probs = analyze_model_performance_gpu(model, test_ds, BATCH_SIZE)

        print("\n✅ TREINAMENTO CONCLUÍDO COM SUCESSO!")
        print(f"📍 Modelos salvos em: {OUTPUT_DIR}")
        print("🎯 Pronto para inferência!")

    except Exception as e:
        print(f"❌ Erro durante execução: {e}")
        import traceback
        traceback.print_exc()

⚠️  Nenhuma GPU detectada. Usando CPU
🚀 INICIANDO TREINAMENTO VISION TRANSFORMER OTIMIZADO PARA GPU
📁 CSV: /content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_undersampling_split.csv
📂 Output: /content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized
🔄 Epochs: 50
📦 Batch Size: 128
🚀 INICIANDO CARREGAMENTO DE DADOS OTIMIZADO PARA GPU
📊 Dataset sizes - Train: 7,243, Val: 1,552, Test: 1,553
📁 Criando dataset com 7243 amostras...
📁 Criando dataset com 1552 amostras...
📁 Criando dataset com 1553 amostras...
⚖️ CALCULANDO CLASS WEIGHTS...
Class weights: {0: np.float64(1.000138083402375), 1: np.float64(0.9998619547211486)}
🧠 CRIANDO MODELO VISION TRANSFORMER OTIMIZADO PARA GPU...
📈 RESUMO DO MODELO:


Model: "vit_gpu_optimized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 5000, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, 26, 256)        │       621,568 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_0             │ (None, 26, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 26, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 26, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 26, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_norm (LayerNormalization) │ (None, 26, 256)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cls_token_extractor (Lambda)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout1 (Dropout)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense1 (Dense)             │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout2 (Dropout)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense2 (Dense)             │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout3 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,829,442 (10.79 MB)

 Trainable params: 2,829,442 (10.79 MB)

 Non-trainable params: 0 (0.00 B)

🎯 INICIANDO TREINAMENTO ACELERADO POR GPU...


KeyboardInterrupt: 

# 8 -Modelo Transformers GPU Otimizada Salvando as epocas.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import wfdb
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import datetime

In [ ]:
# ===========================================
# 0️⃣ CONFIGURAÇÃO GPU E PERFORMANCE
# ===========================================

# Configurações para máxima performance na GPU
def setup_gpu():
    """Configura TensorFlow para máxima performance na GPU"""
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            # Permite crescimento de memória para evitar OOM
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)

            # Configuração de estratégia para multi-GPU se disponível
            if len(gpus) > 1:
                strategy = tf.distribute.MirroredStrategy()
                print(f'🚀 {len(gpus)} GPUs detectadas. Usando estratégia MirroredStrategy')
                return strategy
            else:
                strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
                print('🚀 1 GPU detectada. Configurando para máxima performance')
                return strategy

        except RuntimeError as e:
            print(f'❌ Erro na configuração GPU: {e}')

    print('⚠️  Nenhuma GPU detectada. Usando CPU')
    return tf.distribute.OneDeviceStrategy(device="/cpu:0")

# Configurar GPU no início
strategy = setup_gpu()

🚀 1 GPU detectada. Configurando para máxima performance


In [ ]:
# ===========================================
# 1️⃣ FUNÇÕES DE CARREGAMENTO OTIMIZADAS PARA GPU
# ===========================================

def read_ecg_record_py(path):
    """Função Python para leitura ECG"""
    try:
        path = path.decode('utf-8').replace("b'", "").replace("'", "").strip()

        if not os.path.exists(path) or not os.path.exists(path.replace('.dat', '.hea')):
            return np.zeros((5000, 12), dtype=np.float32)

        record = wfdb.rdrecord(path.replace('.dat', ''))
        signal = record.p_signal.astype(np.float32)

        # Garante 12 derivações
        if signal.shape[1] != 12:
            if signal.shape[1] < 12:
                padding = np.zeros((signal.shape[0], 12 - signal.shape[1]), dtype=np.float32)
                signal = np.hstack([signal, padding])
            else:
                signal = signal[:, :12]

        # Garante 5000 amostras
        if signal.shape[0] != 5000:
            if signal.shape[0] < 5000:
                padding = np.zeros((5000 - signal.shape[0], 12), dtype=np.float32)
                signal = np.vstack([signal, padding])
            else:
                signal = signal[:5000, :]

        # Normalização rápida
        min_val = np.min(signal, axis=0)
        max_val = np.max(signal, axis=0)
        range_val = np.maximum(max_val - min_val, 1e-8)  # Evita divisão por zero

        signal_norm = (signal - min_val) / range_val
        return np.nan_to_num(signal_norm, nan=0.0, posinf=1.0, neginf=0.0)

    except Exception as e:
        return np.zeros((5000, 12), dtype=np.float32)

def create_optimized_dataset(df, batch_size=64, shuffle=True, cache=True):
    """Cria dataset otimizado para GPU com pipeline eficiente"""
    paths = df['dat_path'].values
    labels = df['chagas_label'].values

    print(f"📁 Criando dataset com {len(paths)} amostras...")

    # Dataset básico
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    # Shuffle primeiro (mais eficiente)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, len(paths)), reshuffle_each_iteration=True)

    # Mapeamento paralelizado
    def _load_and_preprocess(path, label):
        signal = tf.numpy_function(read_ecg_record_py, [path], tf.float32)
        signal = tf.ensure_shape(signal, [5000, 12])
        return signal, tf.one_hot(label, depth=2)

    ds = ds.map(_load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch
    ds = ds.batch(batch_size)

    # Cache se possível (para datasets que cabem na memória)
    if cache:
        ds = ds.cache()

    # Prefetch para sobrepor computação de dados e treinamento
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

def load_split_datasets_gpu(csv_path, batch_size=64):
    """Carrega datasets otimizados para GPU"""
    df = pd.read_csv(csv_path)

    train_df = df[df['set'] == 'train']
    val_df = df[df['set'] == 'val']
    test_df = df[df['set'] == 'test']

    print(f"📊 Dataset sizes - Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

    # Usar cache apenas para treino (val e test não precisam)
    train_ds = create_optimized_dataset(train_df, batch_size, shuffle=True, cache=True)
    val_ds = create_optimized_dataset(val_df, batch_size, shuffle=False, cache=False)
    test_ds = create_optimized_dataset(test_df, batch_size, shuffle=False, cache=False)

    return train_ds, val_ds, test_ds

In [ ]:
# ===========================================
# 2️⃣ ARQUITETURA VISION TRANSFORMER
# ===========================================

class PatchEmbedding(layers.Layer):
    """Layer para embeddings de patches com CLS token"""
    def __init__(self, patch_size, hidden_size, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.hidden_size = hidden_size
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.patch_embeddings = layers.Conv1D(
            filters=self.hidden_size,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            use_bias=True,
            name="patch_conv"
        )

        # CLS token como weight
        self.cls_token = self.add_weight(
            shape=(1, 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="cls_token"
        )

        # Positional embeddings
        num_patches = input_shape[1] // self.patch_size
        self.position_embeddings = self.add_weight(
            shape=(1, num_patches + 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="position_embeddings"
        )

        self.dropout = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs):
        # Gerar patches
        patches = self.patch_embeddings(inputs)

        # Adicionar CLS token
        batch_size = tf.shape(inputs)[0]
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        embeddings = tf.concat([cls_tokens, patches], axis=1)

        # Adicionar positional embeddings
        embeddings = embeddings + self.position_embeddings

        return self.dropout(embeddings)

    def get_config(self):
        config = super().get_config()
        config.update({
            'patch_size': self.patch_size,
            'hidden_size': self.hidden_size,
            'dropout_rate': self.dropout_rate,
        })
        return config

class TransformerBlock(layers.Layer):
    """Bloco Transformer otimizado para GPU"""
    def __init__(self, num_heads, hidden_size, mlp_ratio=2, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.hidden_size = hidden_size
        self.mlp_ratio = mlp_ratio
        self.dropout_rate = dropout

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attention = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.hidden_size // self.num_heads,
            dropout=self.dropout_rate,
            use_bias=True
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        # MLP
        self.mlp_dense1 = layers.Dense(self.hidden_size * self.mlp_ratio, activation='gelu')
        self.mlp_dropout1 = layers.Dropout(self.dropout_rate)
        self.mlp_dense2 = layers.Dense(self.hidden_size)
        self.mlp_dropout2 = layers.Dropout(self.dropout_rate)

        self.dropout = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        # Attention com skip connection
        x_norm = self.norm1(inputs, training=training)
        attn_output = self.attention(x_norm, x_norm, training=training)
        x = inputs + self.dropout(attn_output, training=training)

        # MLP com skip connection
        x_norm2 = self.norm2(x, training=training)
        mlp_output = self.mlp_dense1(x_norm2)
        mlp_output = self.mlp_dropout1(mlp_output, training=training)
        mlp_output = self.mlp_dense2(mlp_output)
        mlp_output = self.mlp_dropout2(mlp_output, training=training)

        return x + mlp_output

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_heads': self.num_heads,
            'hidden_size': self.hidden_size,
            'mlp_ratio': self.mlp_ratio,
            'dropout_rate': self.dropout_rate,
        })
        return config

In [ ]:
# ===========================================
# 3️⃣ REGISTRO DAS LAYERS CUSTOMIZADAS
# ===========================================

# Dicionário com todas as layers customizadas para carregamento
CUSTOM_OBJECTS = {
    'PatchEmbedding': PatchEmbedding,
    'TransformerBlock': TransformerBlock
}

def create_vit_model_gpu_optimized(
    input_shape=(5000, 12),
    patch_size=100,
    hidden_size=384,
    depth=6,
    num_heads=8,
    mlp_ratio=2,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """Cria modelo ViT otimizado para GPU - VERSÃO CORRIGIDA"""

    with strategy.scope():
        # Input
        inputs = layers.Input(shape=input_shape, name="ecg_input")

        # Patch Embedding com CLS token
        x = PatchEmbedding(
            patch_size=patch_size,
            hidden_size=hidden_size,
            dropout=dropout,
            name="patch_embedding"
        )(inputs)

        # Transformer Blocks
        for i in range(depth):
            x = TransformerBlock(
                num_heads=num_heads,
                hidden_size=hidden_size,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                name=f"transformer_block_{i}"
            )(x)

        # Classification Head
        x = layers.LayerNormalization(epsilon=1e-6, name="final_norm")(x)

        # Pegar apenas o token CLS (primeira posição)
        cls_output = layers.Lambda(lambda x: x[:, 0], name="cls_token_extractor")(x)

        # MLP Head
        x = layers.Dropout(dropout, name="head_dropout1")(cls_output)
        x = layers.Dense(256, activation='gelu', kernel_initializer='he_normal', name="head_dense1")(x)
        x = layers.Dropout(dropout, name="head_dropout2")(x)
        x = layers.Dense(128, activation='gelu', kernel_initializer='he_normal', name="head_dense2")(x)
        x = layers.Dropout(dropout, name="head_dropout3")(x)

        outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name="vit_gpu_optimized")

        # Otimizador
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=0.01,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-8
        )

        # Compilar
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.AUC(name='auc', curve='ROC'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.F1Score(name='f1_score')
            ]
        )

    return model

In [ ]:
# ===========================================
# 4️⃣ CALLBACKS PARA SALVAR MODELOS POR ÉPOCA
# ===========================================

class EpochModelCheckpoint(tf.keras.callbacks.Callback):
    """Callback para salvar modelo a cada época com nome organizado"""

    def __init__(self, output_dir, save_freq=1, verbose=1):
        super().__init__()
        self.output_dir = output_dir
        self.save_freq = save_freq
        self.verbose = verbose
        self.epochs_dir = os.path.join(output_dir, "epoch_checkpoints")
        os.makedirs(self.epochs_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_freq == 0:
            # Métricas atuais
            val_auc = logs.get('val_auc', 0)
            val_accuracy = logs.get('val_accuracy', 0)
            val_loss = logs.get('val_loss', 0)

            # Nome do arquivo organizado
            filename = f"epoch_{epoch+1:03d}_auc_{val_auc:.4f}_acc_{val_accuracy:.4f}_loss_{val_loss:.4f}.h5"
            filepath = os.path.join(self.epochs_dir, filename)

            # Salvar modelo
            self.model.save(filepath)

            if self.verbose > 0:
                print(f"💾 Modelo da época {epoch+1} salvo como: {filename}")

class BestModelCheckpoint(tf.keras.callbacks.Callback):
    """Callback para salvar o melhor modelo baseado em val_auc"""

    def __init__(self, output_dir, verbose=1):
        super().__init__()
        self.output_dir = output_dir
        self.verbose = verbose
        self.best_auc = 0
        self.best_epoch = 0
        self.best_models_dir = os.path.join(output_dir, "best_models")
        os.makedirs(self.best_models_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        current_auc = logs.get('val_auc', 0)

        if current_auc > self.best_auc:
            self.best_auc = current_auc
            self.best_epoch = epoch + 1

            # Nome do arquivo para o melhor modelo
            filename = f"BEST_epoch_{epoch+1}_auc_{current_auc:.4f}.h5"
            filepath = os.path.join(self.best_models_dir, filename)

            # Salvar modelo
            self.model.save(filepath)

            # Remover modelo anterior se existir
            self._clean_old_models()

            if self.verbose > 0:
                print(f"🏆 NOVO MELHOR MODELO! Época {epoch+1}, AUC: {current_auc:.4f}")
                print(f"💾 Salvo como: {filename}")

    def _clean_old_models(self):
        """Remove modelos antigos da pasta best_models"""
        best_files = [f for f in os.listdir(self.best_models_dir) if f.startswith("BEST_")]
        if len(best_files) > 3:  # Manter apenas os 3 melhores
            best_files.sort()
            for old_file in best_files[:-3]:
                os.remove(os.path.join(self.best_models_dir, old_file))

def get_comprehensive_callbacks(output_dir):
    """Retorna todos os callbacks para treinamento completo"""

    # Criar diretórios
    os.makedirs(output_dir, exist_ok=True)

    # Timestamp para organização
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    callbacks = [
        # 1. Salvar modelo a cada época
        EpochModelCheckpoint(
            output_dir=output_dir,
            save_freq=1,  # Salvar toda época
            verbose=1
        ),

        # 2. Salvar melhor modelo baseado em val_auc
        BestModelCheckpoint(
            output_dir=output_dir,
            verbose=1
        ),

        # 3. Checkpoint tradicional (compatibilidade)
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(output_dir, f"checkpoint_{timestamp}.h5"),
            monitor='val_auc',
            save_best_only=True,
            save_weights_only=False,
            mode='max',
            verbose=1
        ),

        # 4. Early Stopping
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            patience=12,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),

        # 5. Redução de LR
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),

        # 6. CSV Logger
        tf.keras.callbacks.CSVLogger(
            filename=os.path.join(output_dir, f"training_log_{timestamp}.csv")
        )
    ]

    return callbacks

In [ ]:
# ===========================================
# 5️⃣ TREINAMENTO OTIMIZADO PARA GPU
# ===========================================

def safe_model_load(checkpoint_path):
    """Carrega modelo de forma segura com layers customizadas"""
    try:
        return tf.keras.models.load_model(
            checkpoint_path,
            custom_objects=CUSTOM_OBJECTS
        )
    except Exception as e:
        print(f"❌ Erro ao carregar modelo: {e}")
        raise

def train_vit_model_gpu(csv_path, output_dir, epochs=150, batch_size=64):
    """Treinamento otimizado para GPU com salvamento por época"""

    # Criar diretório principal
    os.makedirs(output_dir, exist_ok=True)

    print("🚀 INICIANDO CARREGAMENTO DE DADOS OTIMIZADO PARA GPU")
    print("=" * 60)

    # Carregar datasets otimizados
    train_ds, val_ds, test_ds = load_split_datasets_gpu(csv_path, batch_size=batch_size)

    # Class weights
    print("⚖️ CALCULANDO CLASS WEIGHTS...")
    df = pd.read_csv(csv_path)
    train_df = df[df['set'] == 'train']
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['chagas_label']),
        y=train_df['chagas_label']
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    print(f"Class weights: {class_weight_dict}")

    # Criar modelo otimizado para GPU
    print("🧠 CRIANDO MODELO VISION TRANSFORMER OTIMIZADO PARA GPU...")
    model = create_vit_model_gpu_optimized(
        hidden_size=256,
        depth=4,
        num_heads=4,
        mlp_ratio=2,
        dropout=0.2,
        learning_rate=0.001
    )

    # Callbacks completos
    callbacks = get_comprehensive_callbacks(output_dir)

    print("📈 RESUMO DO MODELO:")
    model.summary()

    # Salvar arquivo com definições das layers customizadas
    custom_objects_path = f"{output_dir}/custom_objects.py"
    with open(custom_objects_path, 'w') as f:
        f.write("""
# Custom objects para carregamento do modelo
CUSTOM_OBJECTS = {
    'PatchEmbedding': PatchEmbedding,
    'TransformerBlock': TransformerBlock
}
""")
    print(f"💾 Definições das layers salvas em: {custom_objects_path}")

    # Informações sobre organização dos modelos
    print("\n📁 ESTRUTURA DE ARQUIVOS:")
    print(f"   📂 {output_dir}/")
    print(f"   ├── 📁 epoch_checkpoints/    (Modelos de cada época)")
    print(f"   ├── 📁 best_models/          (Melhores modelos)")
    print(f"   ├── custom_objects.py        (Definições das layers)")
    print(f"   └── training_log_*.csv       (Logs de treinamento)")
    print("\n🎯 INICIANDO TREINAMENTO ACELERADO POR GPU...")
    print("=" * 60)

    # Treinamento
    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    # Encontrar o melhor modelo salvo
    best_models_dir = os.path.join(output_dir, "best_models")
    best_files = [f for f in os.listdir(best_models_dir) if f.startswith("BEST_")]

    if best_files:
        best_files.sort()
        best_model_file = os.path.join(best_models_dir, best_files[-1])
        print(f"📥 CARREGANDO MELHOR MODELO: {best_files[-1]}")

        try:
            best_model = safe_model_load(best_model_file)
            print("✅ Melhor modelo carregado com sucesso!")
        except Exception as e:
            print(f"⚠️  Usando modelo atual devido a erro: {e}")
            best_model = model
    else:
        print("⚠️  Nenhum melhor modelo encontrado. Usando modelo atual.")
        best_model = model

    # Avaliação final
    print("📊 AVALIAÇÃO FINAL...")
    test_results = best_model.evaluate(test_ds, verbose=1, return_dict=True)

    print("\n🎉 RESULTADOS FINAIS:")
    print("=" * 40)
    for metric, value in test_results.items():
        print(f"{metric:12}: {value:.4f}")

    # Salvar modelo final
    final_path = f"{output_dir}/FINAL_MODEL.h5"
    best_model.save(final_path)
    print(f"\n💾 Modelo final salvo em: {final_path}")

    # Estatísticas dos modelos salvos
    epoch_checkpoints_dir = os.path.join(output_dir, "epoch_checkpoints")
    epoch_files = [f for f in os.listdir(epoch_checkpoints_dir) if f.endswith('.h5')]

    print(f"\n📊 ESTATÍSTICAS DOS MODELOS SALVOS:")
    print(f"   • Modelos por época: {len(epoch_files)}")
    print(f"   • Melhores modelos: {len(best_files)}")
    print(f"   • Épocas treinadas: {len(history.history['loss'])}")

    return best_model, history, test_results


In [ ]:
# ===========================================
# 6️⃣ ANÁLISE DE PERFORMANCE ACELERADA
# ===========================================

def analyze_model_performance_gpu(model, test_ds, batch_size=128):
    """Análise de performance otimizada para GPU"""
    print("🔍 ANALISANDO PERFORMANCE DO MODELO...")

    # Predição em batch para GPU
    y_pred_proba = model.predict(test_ds, batch_size=batch_size, verbose=1)

    # Coletar labels verdadeiros
    y_true = np.concatenate([y for x, y in test_ds], axis=0)
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_pred_proba, axis=1)

    print("\n📋 RELATÓRIO DE CLASSIFICAÇÃO:")
    print("=" * 50)
    print(classification_report(y_true_labels, y_pred, target_names=['Não Chagas', 'Chagas']))

    print("\n🎯 MATRIZ DE CONFUSÃO:")
    print("=" * 30)
    cm = confusion_matrix(y_true_labels, y_pred)
    print(cm)

    auc_score = roc_auc_score(y_true_labels, y_pred_proba[:, 1])
    print(f"\n📊 AUC Score: {auc_score:.4f}")

    return y_true_labels, y_pred, y_pred_proba

In [ ]:
# ===========================================
# 7️⃣ SCRIPT PRINCIPAL OTIMIZADO
# ===========================================

if __name__ == "__main__":
    # Configurações otimizadas para GPU
    CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized'
    EPOCHS = 60
    BATCH_SIZE = 128

    try:
        print("🚀 INICIANDO TREINAMENTO VISION TRANSFORMER OTIMIZADO PARA GPU")
        print("=" * 70)
        print(f"📁 CSV: {CSV_PATH}")
        print(f"📂 Output: {OUTPUT_DIR}")
        print(f"🔄 Epochs: {EPOCHS}")
        print(f"📦 Batch Size: {BATCH_SIZE}")
        print("=" * 70)

        # Treinamento otimizado para GPU
        model, history, test_results = train_vit_model_gpu(
            csv_path=CSV_PATH,
            output_dir=OUTPUT_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE
        )

        # Análise de performance
        print("\n" + "=" * 70)
        print("📊 ANÁLISE DETALHADA DE PERFORMANCE")
        print("=" * 70)

        _, _, test_ds = load_split_datasets_gpu(CSV_PATH, batch_size=BATCH_SIZE)
        y_true, y_pred, y_probs = analyze_model_performance_gpu(model, test_ds, BATCH_SIZE)

        print("\n✅ TREINAMENTO CONCLUÍDO COM SUCESSO!")
        print(f"📍 Modelos salvos em: {OUTPUT_DIR}")
        print("🎯 Pronto para inferência!")

    except Exception as e:
        print(f"❌ Erro durante execução: {e}")
        import traceback
        traceback.print_exc()

🚀 INICIANDO TREINAMENTO VISION TRANSFORMER OTIMIZADO PARA GPU
📁 CSV: /content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split.csv
📂 Output: /content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized
🔄 Epochs: 60
📦 Batch Size: 128
🚀 INICIANDO CARREGAMENTO DE DADOS OTIMIZADO PARA GPU
📊 Dataset sizes - Train: 342,104, Val: 73,308, Test: 73,308
📁 Criando dataset com 342104 amostras...
📁 Criando dataset com 73308 amostras...
📁 Criando dataset com 73308 amostras...
⚖️ CALCULANDO CLASS WEIGHTS...
Class weights: {0: np.float64(1.0), 1: np.float64(1.0)}
🧠 CRIANDO MODELO VISION TRANSFORMER OTIMIZADO PARA GPU...
📈 RESUMO DO MODELO:


Model: "vit_gpu_optimized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 5000, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, 51, 256)        │       320,768 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_0             │ (None, 51, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 51, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 51, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 51, 256)        │       527,104 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_norm (LayerNormalization) │ (None, 51, 256)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cls_token_extractor (Lambda)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout1 (Dropout)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense1 (Dense)             │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout2 (Dropout)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense2 (Dense)             │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout3 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,528,642 (9.65 MB)

 Trainable params: 2,528,642 (9.65 MB)

 Non-trainable params: 0 (0.00 B)

💾 Definições das layers salvas em: /content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized/custom_objects.py

📁 ESTRUTURA DE ARQUIVOS:
   📂 /content/drive/MyDrive/Doutorado-Material/Models/ViT_Chagas_GPU_Optimized/
   ├── 📁 epoch_checkpoints/    (Modelos de cada época)
   ├── 📁 best_models/          (Melhores modelos)
   ├── custom_objects.py        (Definições das layers)
   └── training_log_*.csv       (Logs de treinamento)

🎯 INICIANDO TREINAMENTO ACELERADO POR GPU...
Epoch 1/60
  67/2673 ━━━━━━━━━━━━━━━━━━━━ 57:47:05 80s/step - accuracy: 0.5006 - auc: 0.5005 - f1_score: 0.4999 - loss: 0.8870 - precision: 0.5006 - recall: 0.5006

KeyboardInterrupt: 

# 9 - Modelo LSTM, XLSTM e GRU

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

class XLSTMLayer(layers.Layer):
    """
    Implementação da XLSTM baseada no paper: https://arxiv.org/pdf/2405.04517
    Combina LSTM tradicional com mecanismos de atenção
    """
    def __init__(self, units, return_sequences=False, dropout=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences
        self.dropout_rate = dropout

    def build(self, input_shape):
        # LSTM tradicional
        self.lstm = layers.LSTM(
            self.units,
            return_sequences=True,  # Sempre True para attention
            dropout=self.dropout_rate,
            name=f"{self.name}_lstm"
        )

        # Mecanismo de atenção
        self.attention = layers.MultiHeadAttention(
            num_heads=4,
            key_dim=self.units // 4,
            dropout=self.dropout_rate,
            name=f"{self.name}_attention"
        )

        # Normalizations
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        # Feed forward
        self.ffn = tf.keras.Sequential([
            layers.Dense(self.units * 2, activation='gelu'),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.units),
            layers.Dropout(self.dropout_rate)
        ], name=f"{self.name}_ffn")

        self.dropout_layer = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        # LSTM layer
        lstm_out = self.lstm(inputs, training=training)
        lstm_out = self.dropout_layer(lstm_out, training=training)

        # Self-attention no output do LSTM
        attention_out = self.attention(
            lstm_out, lstm_out, training=training
        )

        # Skip connection + layer norm
        x = self.layernorm1(lstm_out + attention_out, training=training)

        # Feed forward
        ffn_out = self.ffn(x, training=training)

        # Skip connection + layer norm
        x = self.layernorm2(x + ffn_out, training=training)

        if not self.return_sequences:
            # Pegar apenas a última saída temporal
            x = x[:, -1, :]

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'units': self.units,
            'return_sequences': self.return_sequences,
            'dropout_rate': self.dropout_rate,
        })
        return config

In [ ]:
class OptimizedLSTM(layers.Layer):
    """
    LSTM otimizada com múltiplas camadas e regularização
    """
    def __init__(self, units, num_layers=2, dropout=0.2, bidirectional=False, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.num_layers = num_layers
        self.dropout_rate = dropout
        self.bidirectional = bidirectional

    def build(self, input_shape):
        self.lstm_layers = []

        for i in range(self.num_layers):
            return_sequences = (i < self.num_layers - 1)  # True para todas exceto última

            lstm_layer = layers.LSTM(
                self.units,
                return_sequences=return_sequences,
                dropout=self.dropout_rate,
                recurrent_dropout=self.dropout_rate * 0.5,  # Dropout recorrente menor
                name=f"{self.name}_lstm_{i}"
            )

            if self.bidirectional:
                lstm_layer = layers.Bidirectional(
                    lstm_layer,
                    name=f"{self.name}_bilstm_{i}"
                )

            self.lstm_layers.append(lstm_layer)

        self.layer_norm = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_layer = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = inputs

        for i, lstm_layer in enumerate(self.lstm_layers):
            x = lstm_layer(x, training=training)

            # Aplicar layer norm e dropout entre camadas LSTM
            if i < len(self.lstm_layers) - 1:
                x = self.layer_norm(x, training=training)
                x = self.dropout_layer(x, training=training)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'units': self.units,
            'num_layers': self.num_layers,
            'dropout_rate': self.dropout_rate,
            'bidirectional': self.bidirectional,
        })
        return config

In [ ]:
class TCNGRU(layers.Layer):
    """
    Combinação GRU + TCN baseada em: https://unit8.com/resources/temporal-convolutional-networks-and-forecasting/
    """
    def __init__(self, units, num_layers=2, dropout=0.2, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.num_layers = num_layers
        self.dropout_rate = dropout
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Camadas TCN (Temporal Convolutional Network)
        self.tcn_layers = []
        for i in range(2):  # 2 camadas TCN
            tcn_layer = tf.keras.Sequential([
                layers.Conv1D(
                    filters=self.units,
                    kernel_size=self.kernel_size,
                    padding='causal',  # Padding causal para preservar ordem temporal
                    dilation_rate=2**i,  # Dilatação exponencial
                    activation='relu',
                    name=f"{self.name}_tcn_conv_{i}"
                ),
                layers.BatchNormalization(name=f"{self.name}_tcn_bn_{i}"),
                layers.Dropout(self.dropout_rate, name=f"{self.name}_tcn_dropout_{i}")
            ])
            self.tcn_layers.append(tcn_layer)

        # Camadas GRU
        self.gru_layers = []
        for i in range(self.num_layers):
            return_sequences = (i < self.num_layers - 1)

            gru_layer = layers.GRU(
                self.units,
                return_sequences=return_sequences,
                dropout=self.dropout_rate,
                recurrent_dropout=self.dropout_rate * 0.5,
                name=f"{self.name}_gru_{i}"
            )
            self.gru_layers.append(gru_layer)

        self.layer_norm = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_layer = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = inputs

        # Aplicar TCN primeiro
        for tcn_layer in self.tcn_layers:
            x_residual = x  # Skip connection
            x = tcn_layer(x, training=training)
            x = x + x_residual  # Residual connection
            x = self.layer_norm(x, training=training)

        # Aplicar GRU
        for i, gru_layer in enumerate(self.gru_layers):
            x = gru_layer(x, training=training)
            if i < len(self.gru_layers) - 1:
                x = self.dropout_layer(x, training=training)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'units': self.units,
            'num_layers': self.num_layers,
            'dropout_rate': self.dropout_rate,
            'kernel_size': self.kernel_size,
        })
        return config

In [ ]:
CUSTOM_OBJECTS = {}

# Atualizar o dicionário de objetos customizados
CUSTOM_OBJECTS.update({
    'XLSTMLayer': XLSTMLayer,
    'OptimizedLSTM': OptimizedLSTM,
    'TCNGRU': TCNGRU
})

def create_rnn_model(
    input_shape=(5000, 12),
    model_type="xlstm",  # "xlstm", "lstm", "gru_tcn"
    units=128,
    num_layers=2,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """Cria modelos RNN otimizados para dados ECG"""

    with strategy.scope():
        inputs = layers.Input(shape=input_shape, name="ecg_input")

        # Camada de pré-processamento opcional
        x = layers.Conv1D(
            filters=64,
            kernel_size=15,
            strides=2,
            activation='relu',
            padding='same',
            name="initial_conv"
        )(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(pool_size=2)(x)
        x = layers.Dropout(dropout)(x)

        # Camada RNN principal
        if model_type == "xlstm":
            x = XLSTMLayer(
                units=units,
                return_sequences=False,
                dropout=dropout,
                name="xlstm_layer"
            )(x)

        elif model_type == "lstm":
            x = OptimizedLSTM(
                units=units,
                num_layers=num_layers,
                dropout=dropout,
                bidirectional=True,  # BiLSTM para melhor performance
                name="optimized_lstm"
            )(x)

        elif model_type == "gru_tcn":
            x = TCNGRU(
                units=units,
                num_layers=num_layers,
                dropout=dropout,
                kernel_size=5,
                name="tcn_gru"
            )(x)

        # Classification Head
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
        x = layers.Dropout(dropout)(x)

        outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name=f"rnn_{model_type}")

        # Otimizador
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=0.01,
            beta_1=0.9,
            beta_2=0.999
        )

        # Compilar
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.AUC(name='auc', curve='ROC'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall')
            ]
        )

    return model

In [ ]:
def train_rnn_models_comparison(csv_path, output_dir, epochs=50, batch_size=128):
    """Treina e compara diferentes arquiteturas RNN"""

    # Criar diretório
    os.makedirs(output_dir, exist_ok=True)

    print("🚀 CARREGANDO DADOS PARA COMPARAÇÃO RNN...")
    train_ds, val_ds, test_ds = load_split_datasets_gpu(csv_path, batch_size=batch_size)

    # Arquiteturas para testar
    models_config = [
        {
            'type': 'xlstm',
            'name': 'XLSTM',
            'units': 128,
            'num_layers': 2,
            'description': 'XLSTM com Atenção'
        },
        {
            'type': 'lstm',
            'name': 'BiLSTM',
            'units': 128,
            'num_layers': 2,
            'description': 'LSTM Bidirectional'
        },
        {
            'type': 'gru_tcn',
            'name': 'TCN-GRU',
            'units': 128,
            'num_layers': 2,
            'description': 'GRU com TCN'
        }
    ]

    results = {}
    best_model = None
    best_auc = 0

    for config in models_config:
        print(f"\n🧪 TREINANDO: {config['description']}")
        print("=" * 50)

        # Criar modelo
        model = create_rnn_model(
            model_type=config['type'],
            units=config['units'],
            num_layers=config['num_layers'],
            dropout=0.2,
            learning_rate=0.001
        )

        # Callbacks
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_auc',
                patience=15,
                restore_best_weights=True,
                mode='max',
                verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=8,
                min_lr=1e-7,
                verbose=1
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=os.path.join(output_dir, f"best_{config['type']}.h5"),
                monitor='val_auc',
                save_best_only=True,
                mode='max',
                verbose=1
            )
        ]

        # Treinar
        history = model.fit(
            train_ds,
            epochs=epochs,
            validation_data=val_ds,
            callbacks=callbacks,
            verbose=1
        )

        # Avaliar
        test_results = model.evaluate(test_ds, verbose=0, return_dict=True)

        # Salvar resultados
        results[config['type']] = {
            'model': model,
            'history': history.history,
            'test_results': test_results,
            'config': config
        }

        print(f"✅ {config['description']} - Test AUC: {test_results['auc']:.4f}")

        # Atualizar melhor modelo
        if test_results['auc'] > best_auc:
            best_auc = test_results['auc']
            best_model = model

    # Salvar melhor modelo
    if best_model:
        best_model_path = os.path.join(output_dir, "BEST_RNN_MODEL.h5")
        best_model.save(best_model_path)
        print(f"\n🏆 MELHOR MODELO RNN SALVO: {best_model_path}")
        print(f"📊 MELHOR AUC: {best_auc:.4f}")

    # Gerar relatório comparativo
    generate_comparison_report(results, output_dir)

    return best_model, results

def generate_comparison_report(results, output_dir):
    """Gera relatório comparativo das arquiteturas"""

    print("\n📊 RELATÓRIO COMPARATIVO - ARQUITETURAS RNN")
    print("=" * 60)

    report_data = []
    for model_type, result in results.items():
        config = result['config']
        test_results = result['test_results']

        report_data.append({
            'Arquitetura': config['description'],
            'AUC': f"{test_results['auc']:.4f}",
            'Acurácia': f"{test_results['accuracy']:.4f}",
            'Precisão': f"{test_results['precision']:.4f}",
            'Recall': f"{test_results['recall']:.4f}",
            'Loss': f"{test_results['loss']:.4f}"
        })

        print(f"\n🔍 {config['description']}:")
        print(f"   📈 AUC:       {test_results['auc']:.4f}")
        print(f"   🎯 Acurácia:  {test_results['accuracy']:.4f}")
        print(f"   📏 Precisão:  {test_results['precision']:.4f}")
        print(f"   🔄 Recall:    {test_results['recall']:.4f}")

    # Salvar relatório em CSV
    report_df = pd.DataFrame(report_data)
    report_path = os.path.join(output_dir, "rnn_comparison_report.csv")
    report_df.to_csv(report_path, index=False)
    print(f"\n💾 Relatório salvo em: {report_path}")

In [ ]:
import os

if __name__ == "__main__":
    # Configurações
    CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/Doutorado-Material/Models/RNN_Comparison'

    try:
        print("🧠 COMPARAÇÃO DE ARQUITETURAS RNN PARA ECG")
        print("=" * 60)
        print("📋 Arquiteturas a serem testadas:")
        print("   • XLSTM (eXtended LSTM com atenção)")
        print("   • BiLSTM (LSTM Bidirectional Otimizada)")
        print("   • TCN-GRU (GRU com Temporal Convolutional Network)")
        print("=" * 60)

        best_model, results = train_rnn_models_comparison(
            csv_path=CSV_PATH,
            output_dir=OUTPUT_DIR,
            epochs=60,
            batch_size=16
        )

        print("\n✅ COMPARAÇÃO CONCLUÍDA!")
        print(f"📍 Modelos salvos em: {OUTPUT_DIR}")

    except Exception as e:
        print(f"❌ Erro durante execução: {e}")
        import traceback
        traceback.print_exc()

🧠 COMPARAÇÃO DE ARQUITETURAS RNN PARA ECG
📋 Arquiteturas a serem testadas:
   • XLSTM (eXtended LSTM com atenção)
   • BiLSTM (LSTM Bidirectional Otimizada)
   • TCN-GRU (GRU com Temporal Convolutional Network)
❌ Erro durante execução: name 'train_rnn_models_comparison' is not defined


Traceback (most recent call last):
  File "/tmp/ipython-input-571045876.py", line 17, in <cell line: 0>
    best_model, results = train_rnn_models_comparison(
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'train_rnn_models_comparison' is not defined


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import gc

# ===========================================
# CONFIGURAÇÃO PARA ECONOMIZAR MEMÓRIA
# ===========================================

def setup_memory_optimization():
    """Configurações para economizar memória RAM"""
    tf.keras.backend.clear_session()
    gc.collect()

    # Configurações do TensorFlow para CPU
    tf.config.optimizer.set_jit(False)
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)

    print("🔧 Configurações de memória aplicadas")

# ===========================================
# LAYERS CUSTOMIZADAS CORRIGIDAS
# ===========================================

class CLSTokenLayer(layers.Layer):
    """Adiciona token CLS usando apenas operações Keras"""
    def __init__(self, hidden_size, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            shape=(1, 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="cls_token"
        )
        super().build(input_shape)

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        return tf.concat([cls_tokens, inputs], axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({'hidden_size': self.hidden_size})
        return config

class PositionEmbeddingLayer(layers.Layer):
    """Adiciona positional embeddings"""
    def __init__(self, num_patches, hidden_size, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.hidden_size = hidden_size

    def build(self, input_shape):
        self.position_embeddings = self.add_weight(
            shape=(1, self.num_patches + 1, self.hidden_size),
            initializer='random_normal',
            trainable=True,
            name="position_embeddings"
        )
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.position_embeddings

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_patches': self.num_patches,
            'hidden_size': self.hidden_size
        })
        return config

class CLSExtractorLayer(layers.Layer):
    """Extrai o token CLS (primeira posição)"""
    def call(self, inputs):
        return inputs[:, 0, :]  # Retorna apenas o token CLS

# ===========================================
# MODELOS CORRIGIDOS - SEM TENSORFLOW PURO
# ===========================================

def create_lightweight_vit_model(
    input_shape=(5000, 12),
    patch_size=250,
    hidden_size=96,
    depth=2,
    num_heads=4,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """ViT leve para CPU - VERSÃO CORRIGIDA"""

    inputs = layers.Input(shape=input_shape, name="ecg_input")

    # 1. Patch Embedding com Conv1D
    x = layers.Conv1D(
        filters=hidden_size,
        kernel_size=patch_size,
        strides=patch_size,
        use_bias=True,
        name="patch_conv"
    )(inputs)

    num_patches = input_shape[1] // patch_size

    # 2. Adicionar CLS token usando layer customizada
    x = CLSTokenLayer(hidden_size=hidden_size, name="cls_token")(x)

    # 3. Adicionar positional embeddings
    x = PositionEmbeddingLayer(
        num_patches=num_patches,
        hidden_size=hidden_size,
        name="position_embedding"
    )(x)

    x = layers.Dropout(dropout)(x)

    # 4. Blocos Transformer simplificados
    for i in range(depth):
        # Self-attention com skip connection
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=hidden_size // num_heads,
            dropout=dropout,
            name=f"attention_{i}"
        )(x, x)

        x = layers.Add()([x, attn_output])
        x = layers.LayerNormalization(epsilon=1e-6, name=f"norm1_{i}")(x)

        # MLP com skip connection
        mlp_output = layers.Dense(hidden_size * 2, activation='relu', name=f"mlp1_{i}")(x)
        mlp_output = layers.Dropout(dropout)(mlp_output)
        mlp_output = layers.Dense(hidden_size, name=f"mlp2_{i}")(mlp_output)

        x = layers.Add()([x, mlp_output])
        x = layers.LayerNormalization(epsilon=1e-6, name=f"norm2_{i}")(x)

    # 5. Extrair CLS token
    x = CLSExtractorLayer(name="cls_extractor")(x)

    # 6. Classification Head simplificado
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu', name="head_dense1")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu', name="head_dense2")(x)

    outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="vit_lightweight")

    # Compilar
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

def create_simple_lstm_model(
    input_shape=(5000, 12),
    units=64,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """LSTM simples e eficiente para CPU"""

    inputs = layers.Input(shape=input_shape, name="ecg_input")

    # Downsampling inicial
    x = layers.Conv1D(32, 15, strides=2, activation='relu', padding='same', name="conv1")(inputs)
    x = layers.MaxPooling1D(2, name="pool1")(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(64, 10, strides=2, activation='relu', padding='same', name="conv2")(x)
    x = layers.MaxPooling1D(2, name="pool2")(x)
    x = layers.Dropout(dropout)(x)

    # LSTM
    x = layers.LSTM(units, return_sequences=True, name="lstm1")(x)
    x = layers.LSTM(units, name="lstm2")(x)

    # Classification Head
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu', name="dense1")(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="simple_lstm")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

def create_simple_gru_model(
    input_shape=(5000, 12),
    units=64,
    num_classes=2,
    dropout=0.2,
    learning_rate=0.001
):
    """GRU simples e eficiente para CPU"""

    inputs = layers.Input(shape=input_shape, name="ecg_input")

    # Downsampling inicial
    x = layers.Conv1D(32, 15, strides=2, activation='relu', padding='same', name="conv1")(inputs)
    x = layers.MaxPooling1D(2, name="pool1")(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(64, 10, strides=2, activation='relu', padding='same', name="conv2")(x)
    x = layers.MaxPooling1D(2, name="pool2")(x)
    x = layers.Dropout(dropout)(x)

    # GRU
    x = layers.GRU(units, return_sequences=True, name="gru1")(x)
    x = layers.GRU(units, name="gru2")(x)

    # Classification Head
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu', name="dense1")(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name="outputs")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="simple_gru")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

# ===========================================
# DATASET FUNCTIONS (MESMAS DO ANTERIOR)
# ===========================================

def create_memory_friendly_dataset(df, batch_size=8, shuffle=True):
    """Dataset otimizado para baixo consumo de memória"""
    paths = df['dat_path'].values
    labels = df['chagas_label'].values

    print(f"📁 Criando dataset memory-friendly com {len(paths)} amostras...")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(1000, len(paths)), reshuffle_each_iteration=True)

    def _load_and_preprocess(path, label):
        signal = tf.numpy_function(read_ecg_record_py, [path], tf.float32)
        signal = tf.ensure_shape(signal, [5000, 12])
        return signal, tf.one_hot(label, depth=2)

    ds = ds.map(_load_and_preprocess, num_parallel_calls=2)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)

    return ds

def load_split_datasets_cpu_optimized(csv_path, batch_size=8):
    """Carrega datasets otimizados para CPU com pouca RAM"""
    df = pd.read_csv(csv_path)

    train_df = df[df['set'] == 'train']
    val_df = df[df['set'] == 'val']
    test_df = df[df['set'] == 'test']

    print(f"📊 Dataset sizes - Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

    train_ds = create_memory_friendly_dataset(train_df, batch_size, shuffle=True)
    val_ds = create_memory_friendly_dataset(val_df, batch_size, shuffle=False)
    test_ds = create_memory_friendly_dataset(test_df, batch_size, shuffle=False)

    return train_ds, val_ds, test_ds

# ===========================================
# TREINAMENTO CORRIGIDO
# ===========================================

def train_with_memory_management(csv_path, output_dir, epochs=30, batch_size=8):
    """Treinamento otimizado para baixo consumo de memória - VERSÃO CORRIGIDA"""

    setup_memory_optimization()

    print("🚀 INICIANDO TREINAMENTO OTIMIZADO PARA CPU")
    print("=" * 50)
    print(f"📦 Batch size: {batch_size}")
    print(f"🔄 Epochs: {epochs}")

    # Carregar datasets
    train_ds, val_ds, test_ds = load_split_datasets_cpu_optimized(csv_path, batch_size)

    # Testar modelos
    models_to_test = [
        ("vit", "Vision Transformer Leve"),
        ("lstm", "LSTM Simples"),
        ("gru", "GRU Simples")
    ]

    results = {}

    for model_type, description in models_to_test:
        print(f"\n🧪 TREINANDO: {description}")
        print("-" * 40)

        # Criar modelo
        if model_type == "vit":
            model = create_lightweight_vit_model()
        elif model_type == "lstm":
            model = create_simple_lstm_model()
        else:
            model = create_simple_gru_model()

        print("📋 Resumo do modelo:")
        model.summary()

        # Callbacks
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_auc',
                patience=8,
                restore_best_weights=True,
                mode='max',
                verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            )
        ]

        # Treinar
        try:
            history = model.fit(
                train_ds,
                epochs=min(epochs, 20),
                validation_data=val_ds,
                callbacks=callbacks,
                verbose=1,
                steps_per_epoch=min(1000, len(train_ds))  # Limitar steps por época
            )

            # Avaliar
            test_results = model.evaluate(test_ds, verbose=0, return_dict=True)
            results[model_type] = {
                'model': model,
                'test_results': test_results,
                'description': description
            }

            print(f"✅ {description} - AUC: {test_results['auc']:.4f}")

        except Exception as e:
            print(f"❌ Erro no treinamento: {e}")
            results[model_type] = {'error': str(e)}

        # Limpar memória
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    return results

# ===========================================
# SCRIPT PRINCIPAL
# ===========================================

if __name__ == "__main__":
    CSV_PATH = '/content/drive/MyDrive/Doutorado-Material/Dataset/balanced_final/dataset_chagas_oversampling_split.csv'
    OUTPUT_DIR = '/content/drive/MyDrive/Doutorado-Material/Models/CPU_Optimized'

    try:
        # Treinar com configurações seguras
        results = train_with_memory_management(
            csv_path=CSV_PATH,
            output_dir=OUTPUT_DIR,
            epochs=25,
            batch_size=8
        )

        print("\n✅ TREINAMENTO CONCLUÍDO!")
        print("\n📊 RESULTADOS FINAIS:")
        for model_type, result in results.items():
            if 'test_results' in result:
                print(f"   {result['description']}: AUC = {result['test_results']['auc']:.4f}")

    except Exception as e:
        print(f"❌ Erro geral: {e}")
        import traceback
        traceback.print_exc()

🔧 Configurações de memória aplicadas
🚀 INICIANDO TREINAMENTO OTIMIZADO PARA CPU
📦 Batch size: 8
🔄 Epochs: 25
📊 Dataset sizes - Train: 342,104, Val: 73,308, Test: 73,308
📁 Criando dataset memory-friendly com 342104 amostras...
📁 Criando dataset memory-friendly com 73308 amostras...
📁 Criando dataset memory-friendly com 73308 amostras...

🧪 TREINANDO: Vision Transformer Leve
----------------------------------------
📋 Resumo do modelo:


Model: "vit_lightweight"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ecg_input           │ (None, 5000, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_conv (Conv1D) │ (None, 20, 96)    │    288,096 │ ecg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_token           │ (None, 21, 96)    │         96 │ patch_conv[0][0]  │
│ (CLSTokenLayer)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 21, 96)    │         96 │ cls_token[0][0]   │
│ (PositionEmbedding… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 21, 96)    │          0 │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_0         │ (None, 21, 96)    │     37,248 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 21, 96)    │          0 │ dropout[0][0],    │
│                     │                   │            │ attention_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm1_0             │ (None, 21, 96)    │        192 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp1_0 (Dense)      │ (None, 21, 192)   │     18,624 │ norm1_0[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 21, 192)   │          0 │ mlp1_0[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp2_0 (Dense)      │ (None, 21, 96)    │     18,528 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 21, 96)    │          0 │ norm1_0[0][0],    │
│                     │                   │            │ mlp2_0[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm2_0             │ (None, 21, 96)    │        192 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 21, 96)    │     37,248 │ norm2_0[0][0],    │
│ (MultiHeadAttentio… │                   │            │ norm2_0[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 21, 96)    │          0 │ norm2_0[0][0],    │
│                     │                   │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm1_1             │ (None, 21, 96)    │        192 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp1_1 (Dense)      │ (None, 21, 192)   │     18,624 │ norm1_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 21, 192)   │          0 │ mlp1_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp2_1 (Dense)      │ (None, 21, 96)    │     18,528 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 446,210 (1.70 MB)

 Trainable params: 446,210 (1.70 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 5000, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 2500, 32)       │         5,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 1250, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1250, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 625, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 312, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 312, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru1 (GRU)                      │ (None, 312, 64)        │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru2 (GRU)                      │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,402 (306.26 KB)

 Trainable params: 78,402 (306.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 858s 854ms/step - accuracy: 0.4885 - auc: 0.5050 - loss: 0.6932 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 0.0010
Epoch 2/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 893s 894ms/step - accuracy: 0.5049 - auc: 0.5058 - loss: 0.6932 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 3/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 890s 891ms/step - accuracy: 0.4993 - auc: 0.4955 - loss: 0.6935 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 0.0010
Epoch 4/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 884s 884ms/step - accuracy: 0.4998 - auc: 0.4930 - loss: 0.6932 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 5/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 877s 878ms/step - accuracy: 0.5038 - auc: 0.5005 - loss: 0.6932 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 6/20
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms